# 20_residual_phm_diagnostics.ipynb

Το NB20 ορίζει ένα μεγάλο, ελεγχόμενο πείραμα **forecasting-based residual diagnostics** για PHM-oriented ερμηνεία του WindPower_DigitalTwin project.

Ο στόχος δεν είναι νέα αρχιτεκτονική, ούτε αντικατάσταση των canonical forecasting benchmarks. Το notebook κατασκευάζει ένα condition-monitoring-oriented layer πάνω σε υπάρχουσες ή leakage-safe προβλέψεις, ώστε τα residuals να μετατραπούν σε ενδείξεις:

- persistent deviation from expected forecast behavior,
- early-warning indicators,
- operating-regime mismatch,
- park-level diagnostic ranking,
- manuscript-ready local tables και figures.

Τα αποτελέσματα αντιμετωπίζονται ως evidence για forecasting-based residual diagnostics και όχι ως confirmed fault diagnosis.


## Literature-Motivated PHM Framing

Η σύνδεση με PHM γίνεται μέσα από τη συμπεριφορά των residuals μεταξύ παρατηρούμενης και προβλεπόμενης ισχύος. Τα local papers χρησιμοποιούνται μόνο ως motivation και όχι ως εκτελέσιμα dependencies.

- Το `08_Gijon_2025_Hybrid_Explainable_Betz_Limit_Constraint.pdf` στηρίζει τη λογική ότι residual modeling, explainability και uncertainty μπορούν να βοηθήσουν στην ερμηνεία αποκλίσεων μεταξύ predicted και observed power.
- Το `30_Dhungana_2025_Wind_Power_Forecasting_ML_vs_DL.pdf` συνδέει forecasting με condition monitoring, decision-making και maintenance planning.
- Ένα local Zou/outlier review PDF, όταν υπάρχει στο repository, λειτουργεί ως κίνητρο για το ότι abnormal operating data και outliers μπορούν να επηρεάσουν wind-turbine και wind-farm models.
- Το `18_Vogt_2022_Synthetic_Wind_Dataset_273_Germany.pdf`, όταν υπάρχει, τεκμηριώνει το DaKS/Kassel synthetic wind dataset με πολλά γεωγραφικά κατανεμημένα wind plants.
- Το `03_Pessoa_2025_Mamba_Uncertainty_ProbTSF.pdf` λειτουργεί μόνο ως motivation ότι uncertainty-aware forecasting βοηθά την ερμηνεία forecast confidence. Δεν υλοποιείται Mamba εδώ.

Η ανάλυση παραμένει διαγνωστική και early-warning: δεν υπάρχουν ground-truth fault labels, άρα τα persistent residuals δεν ονομάζονται faults.


## Configuration And Reproducibility

Οι default ρυθμίσεις είναι ασφαλείς για γρήγορο local άνοιγμα: smoke subset, χωρίς full diagnostics και χωρίς exports. Για full local run αλλάζονται χειροκίνητα τα flags στην επόμενη cell.


In [ ]:
# ============================================================
# NB20 | Imports, seeds και βασικές ρυθμίσεις
# ============================================================

from pathlib import Path

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)

import itertools
import re
import subprocess
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from sklearn.ensemble import HistGradientBoostingRegressor
    from sklearn.impute import SimpleImputer
    SKLEARN_AVAILABLE = True
    SKLEARN_IMPORT_ERROR = None
except Exception as exc:
    HistGradientBoostingRegressor = None
    SimpleImputer = None
    SKLEARN_AVAILABLE = False
    SKLEARN_IMPORT_ERROR = exc

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
np.random.seed(SEED)

# Ασφαλείς default ρυθμίσεις.
SMOKE_MODE = True
RUN_FULL_DIAGNOSTICS = False
EXPORT_RESULTS = False
FULL_EXPORT_RESIDUAL_RECORDS = False

# Για χειροκίνητο full local run:
# SMOKE_MODE = False
# RUN_FULL_DIAGNOSTICS = True
# EXPORT_RESULTS = True

SELECTED_SMOKE_PARKS = ["00183", "00198", "00303", "00427"]
FULL_MODE_PARK_LIMIT = None
MIN_PARK_VALIDATION_ROWS = 96
ROLLING_WINDOWS = [24, 72]
ROLLING_MIN_PERIODS = {24: 6, 72: 18}
TOP_N_DISPLAY = 20
SMOKE_OPTIONAL_PREDICTION_MAX_ROWS = 250_000

TARGET_COL = "Power_Output_Normalized"
PARK_COL = "park_id"
TIME_COL = "timestamp"
TEST_FLAG_COL = "test_flag"

PREDICTION_COLUMN_CANDIDATES = [
    "Baseline_Prediction",
    "prediction",
    "y_pred",
    "predicted",
    "model_prediction",
]

METADATA_COLUMNS = [
    "lat",
    "long",
    "hub_height_m",
    "rotor_diameter_m",
    "nominal_power_kW",
]

REGIME_COLUMNS = [
    "nwp_fcst_horiz_hours",
    "U_GVL_58_HL",
    "V_GVL_58_HL",
    "U_GVL_60_HL",
    "V_GVL_60_HL",
    "T_HAG_2_M",
    "RELHUM_HAG_2_M",
    "PS_SFC_0_M",
    "ws_ref",
    "Wind_Speed_100m_ms",
]

FORBIDDEN_OUTPUT_SUFFIXES = {".pkl", ".pickle", ".joblib", ".bin", ".model", ".pt", ".pth", ".ckpt"}

REQUIRED_RESIDUAL_CONTRACT_COLUMNS = [
    "park_id",
    "timestamp",
    "y_true",
    "y_pred",
    "residual",
    "abs_error",
    "squared_error",
    "residual_sign",
    "residual_z_score",
    "robust_residual_z_score",
    "rolling_MAE_24",
    "rolling_MAE_72",
    "rolling_bias_24",
    "rolling_bias_72",
    "warning_abs_q95",
    "warning_robust_z",
    "warning_rolling_mae_24",
    "warning_flag",
    "warning_reason",
]


In [ ]:
# ============================================================
# NB20 | Paths και προστατευμένα canonical artifacts
# ============================================================

def find_project_root(start_path=None):
    '''Εντοπίζει το project root από γνωστούς canonical φακέλους.'''
    current = Path(start_path or Path.cwd()).resolve()
    candidates = [current, *current.parents]
    for candidate in candidates:
        if (candidate / "data" / "processed").exists() and (candidate / "notebooks").exists():
            return candidate
    return current


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "processed"
PREDICTION_DIR = DATA_DIR / "predictions"
PAPERS_DIR = PROJECT_ROOT / "literature_local" / "papers"

TRAIN_PATH = DATA_DIR / "train_final.csv"
VAL_PATH = DATA_DIR / "val_final.csv"
TEST_PATH = DATA_DIR / "test_final.csv"
BASELINE_METRICS_PATH = DATA_DIR / "baseline_metrics.csv"
REQUIREMENTS_PATH = PROJECT_ROOT / "requirements.txt"

OUTPUT_DIR = DATA_DIR / "diagnostics" / "residual_phm_diagnostics"
FIGURE_DIR = OUTPUT_DIR / "figures"

OPTIONAL_PREDICTION_PATHS = {
    "nb06_test_predictions": DATA_DIR / "nb06_test_predictions.csv",
    "nb07_all_test_predictions_long": PREDICTION_DIR / "nb07_all_test_predictions_long.csv",
    "nb07_xgboost_test_predictions": PREDICTION_DIR / "nb07_xgboost_test_predictions.csv",
    "nb07_random_forest_test_predictions": PREDICTION_DIR / "nb07_random_forest_test_predictions.csv",
    "nb07_mlp_test_predictions": PREDICTION_DIR / "nb07_mlp_test_predictions.csv",
}

LOCAL_LITERATURE_PATHS = {
    "gijon_2025": PAPERS_DIR / "08_Gijon_2025_Hybrid_Explainable_Betz_Limit_Constraint.pdf",
    "dhungana_2025": PAPERS_DIR / "30_Dhungana_2025_Wind_Power_Forecasting_ML_vs_DL.pdf",
    "vogt_2022": PAPERS_DIR / "18_Vogt_2022_Synthetic_Wind_Dataset_273_Germany.pdf",
    "pessoa_2025": PAPERS_DIR / "03_Pessoa_2025_Mamba_Uncertainty_ProbTSF.pdf",
}

zoutlier_candidates = []
if PAPERS_DIR.exists():
    zoutlier_candidates = sorted(PAPERS_DIR.glob("*Zou*Outlier*.pdf")) + sorted(PAPERS_DIR.glob("*outlier*review*.pdf"))
if zoutlier_candidates:
    LOCAL_LITERATURE_PATHS["zou_outlier_review"] = zoutlier_candidates[0]

PROTECTED_PATHS = {
    "baseline_metrics": BASELINE_METRICS_PATH,
    "requirements": REQUIREMENTS_PATH,
}

protected_mtimes_before = {
    name: path.stat().st_mtime_ns if path.exists() else None
    for name, path in PROTECTED_PATHS.items()
}


## NB20 Experiment Design Matrix

Ο πίνακας συνδέει το reviewer-facing PHM ζητούμενο με το συγκεκριμένο NB20 diagnostic response. Η έμφαση παραμένει σε leakage-safe residual interpretation και όχι σε νέα αρχιτεκτονική ή confirmed fault diagnosis.


In [ ]:
# ============================================================
# NB20 | Experiment design matrix για manuscript planning
# ============================================================

experiment_design_matrix_df = pd.DataFrame([
    {
        "reviewer_comment_component": "residual analysis",
        "implemented_NB20_diagnostic_response": "validation/test residual records with absolute, squared, signed, standard and robust residual scores",
        "leakage_control": "validation residuals calibrate thresholds; test residuals are interpreted once",
        "manuscript_output": "overall residual metrics, residual distributions, residual-vs-predicted figure",
    },
    {
        "reviewer_comment_component": "condition-monitoring indicators",
        "implemented_NB20_diagnostic_response": "warning flags for high absolute error, robust residual z-score and rolling residual stress",
        "leakage_control": "warning thresholds are derived only from validation residuals",
        "manuscript_output": "threshold policy table and warning-rate summaries",
    },
    {
        "reviewer_comment_component": "park-level diagnostic ranking",
        "implemented_NB20_diagnostic_response": "park summaries ranked by MAE, warning rate and absolute mean residual",
        "leakage_control": "ranking uses fixed test residuals after validation calibration",
        "manuscript_output": "top diagnostic parks tables and bar charts",
    },
    {
        "reviewer_comment_component": "operating-regime diagnostics",
        "implemented_NB20_diagnostic_response": "target, predicted, wind-speed, temperature and forecast-horizon residual bins",
        "leakage_control": "bins describe test behavior only after prediction source and thresholds are fixed",
        "manuscript_output": "operating-regime high-error summary table and figure",
    },
    {
        "reviewer_comment_component": "rolling early-warning indicators",
        "implemented_NB20_diagnostic_response": "rolling MAE, rolling bias and rolling RMSE at 24/72 row windows",
        "leakage_control": "rolling q95 warning threshold is calibrated on validation only",
        "manuscript_output": "rolling residual figure and rolling warning-rate diagnostics",
    },
    {
        "reviewer_comment_component": "warning-event extraction",
        "implemented_NB20_diagnostic_response": "contiguous warning rows are grouped into candidate diagnostic events with severity ranking",
        "leakage_control": "events are extracted on test after validation thresholds are locked",
        "manuscript_output": "warning-event summary and top warning-event severity table",
    },
    {
        "reviewer_comment_component": "optional model comparison",
        "implemented_NB20_diagnostic_response": "compatible NB06/NB07 prediction CSVs are interpreted as residual summaries by model",
        "leakage_control": "optional test CSVs are never used for threshold derivation",
        "manuscript_output": "model residual comparison table when schemas are compatible",
    },
    {
        "reviewer_comment_component": "PHM boundary",
        "implemented_NB20_diagnostic_response": "persistent deviations are framed as forecasting-based residual diagnostics and condition-monitoring-oriented evidence",
        "leakage_control": "no fault labels, no test-based calibration and no canonical benchmark mutation",
        "manuscript_output": "limitations and interpretation-boundary text",
    },
])

display(experiment_design_matrix_df)


## Path And Data Availability Audit

Η audit table επιβεβαιώνει τα canonical splits, τα optional prediction CSVs και τα local literature files που λειτουργούν μόνο ως motivation.


In [ ]:
# ============================================================
# NB20 | Path audit χωρίς μεταλλάξεις αρχείων
# ============================================================

def csv_schema_preview(path, nrows=0):
    '''Διαβάζει μόνο schema preview για ελαφρύ audit.'''
    if not path.exists():
        return []
    try:
        return list(pd.read_csv(path, nrows=nrows).columns)
    except Exception as exc:
        return [f"schema_error: {exc}"]


def canonical_split_path_audit(label, path):
    row = {
        "artifact_group": "canonical_split",
        "label": label,
        "path": str(path.relative_to(PROJECT_ROOT)),
        "exists": path.exists(),
        "columns_preview": ", ".join(csv_schema_preview(path)[:12]),
        "min_timestamp": pd.NaT,
        "max_timestamp": pd.NaT,
        "unique_parks": np.nan,
        "duplicate_park_timestamp_count": np.nan,
        "missing_target_count": np.nan,
        "missing_timestamp_count": np.nan,
        "missing_park_id_count": np.nan,
        "target_min": np.nan,
        "target_max": np.nan,
        "target_mean": np.nan,
        "prediction_columns_discovered": "",
    }
    if not path.exists():
        return row
    try:
        df = pd.read_csv(path, dtype={PARK_COL: "string"})
        if PARK_COL in df.columns:
            normalized_park = df[PARK_COL].map(normalize_park_id_value) if "normalize_park_id_value" in globals() else df[PARK_COL]
            row["unique_parks"] = int(normalized_park.nunique(dropna=True))
            row["missing_park_id_count"] = int(df[PARK_COL].isna().sum())
        if TIME_COL in df.columns:
            timestamps = pd.to_datetime(df[TIME_COL], errors="coerce")
            row["min_timestamp"] = timestamps.min()
            row["max_timestamp"] = timestamps.max()
            row["missing_timestamp_count"] = int(timestamps.isna().sum())
        if {PARK_COL, TIME_COL}.issubset(df.columns):
            duplicate_mask = df.duplicated([PARK_COL, TIME_COL], keep=False)
            row["duplicate_park_timestamp_count"] = int(duplicate_mask.sum())
        if TARGET_COL in df.columns:
            target = pd.to_numeric(df[TARGET_COL], errors="coerce")
            row["missing_target_count"] = int(target.isna().sum())
            row["target_min"] = float(target.min())
            row["target_max"] = float(target.max())
            row["target_mean"] = float(target.mean())
        discovered = [col for col in PREDICTION_COLUMN_CANDIDATES if col in df.columns]
        row["prediction_columns_discovered"] = ", ".join(discovered)
    except Exception as exc:
        row["columns_preview"] = f"path_audit_error: {exc}"
    return row


path_audit_rows = []
for label, path in {
    "train_final": TRAIN_PATH,
    "val_final": VAL_PATH,
    "test_final": TEST_PATH,
}.items():
    path_audit_rows.append(canonical_split_path_audit(label, path))

for label, path in OPTIONAL_PREDICTION_PATHS.items():
    path_audit_rows.append({
        "artifact_group": "optional_prediction_csv",
        "label": label,
        "path": str(path.relative_to(PROJECT_ROOT)),
        "exists": path.exists(),
        "columns_preview": ", ".join(csv_schema_preview(path)[:12]),
        "min_timestamp": pd.NaT,
        "max_timestamp": pd.NaT,
        "unique_parks": np.nan,
        "duplicate_park_timestamp_count": np.nan,
        "missing_target_count": np.nan,
        "missing_timestamp_count": np.nan,
        "missing_park_id_count": np.nan,
        "target_min": np.nan,
        "target_max": np.nan,
        "target_mean": np.nan,
        "prediction_columns_discovered": "",
    })

for label, path in LOCAL_LITERATURE_PATHS.items():
    path_audit_rows.append({
        "artifact_group": "local_literature_motivation",
        "label": label,
        "path": str(path.relative_to(PROJECT_ROOT)) if path.exists() else str(path),
        "exists": path.exists(),
        "columns_preview": "not_applicable",
        "min_timestamp": pd.NaT,
        "max_timestamp": pd.NaT,
        "unique_parks": np.nan,
        "duplicate_park_timestamp_count": np.nan,
        "missing_target_count": np.nan,
        "missing_timestamp_count": np.nan,
        "missing_park_id_count": np.nan,
        "target_min": np.nan,
        "target_max": np.nan,
        "target_mean": np.nan,
        "prediction_columns_discovered": "",
    })

path_audit_df = pd.DataFrame(path_audit_rows)
display(path_audit_df)

missing_required_paths = path_audit_df.query("artifact_group == 'canonical_split' and exists == False")
if not missing_required_paths.empty:
    raise FileNotFoundError("Missing required canonical split files: " + ", ".join(missing_required_paths["label"].tolist()))


## Canonical Split Loading

Τα canonical splits φορτώνονται χωρίς αλλαγή στα upstream artifacts. Το `park_id` διατηρείται ως zero-padded string, το `timestamp` γίνεται datetime και οι γραμμές ταξινομούνται ανά park και χρόνο.


In [ ]:
# ============================================================
# NB20 | Φόρτωση canonical splits και smoke/full subset policy
# ============================================================

def normalize_park_id_value(value, width=5):
    '''Μετατρέπει numeric ή string park ids σε σταθερό zero-padded format.'''
    if pd.isna(value):
        return pd.NA
    text = str(value).strip()
    if re.fullmatch(r"\d+(\.0+)?", text):
        text = str(int(float(text)))
    return text.zfill(width)


def load_canonical_split(path, split_name):
    df = pd.read_csv(path, dtype={PARK_COL: "string"})
    required_cols = {PARK_COL, TIME_COL, TARGET_COL}
    missing = sorted(required_cols - set(df.columns))
    if missing:
        raise ValueError(f"{split_name} missing required columns: {missing}")
    df[PARK_COL] = df[PARK_COL].map(normalize_park_id_value).astype("string")
    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
    df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
    df = df.sort_values([PARK_COL, TIME_COL]).reset_index(drop=True)
    df["split"] = split_name
    return df


def apply_working_subset(df):
    '''Εφαρμόζει deterministic subset μόνο όταν ζητείται smoke run ή explicit full limit.'''
    if SMOKE_MODE and not RUN_FULL_DIAGNOSTICS:
        selected = df[df[PARK_COL].isin(SELECTED_SMOKE_PARKS)].copy()
        if selected.empty:
            fallback_parks = sorted(df[PARK_COL].dropna().unique().tolist())[: len(SELECTED_SMOKE_PARKS)]
            selected = df[df[PARK_COL].isin(fallback_parks)].copy()
        return selected.reset_index(drop=True)

    if FULL_MODE_PARK_LIMIT is not None:
        selected_parks = sorted(df[PARK_COL].dropna().unique().tolist())[: int(FULL_MODE_PARK_LIMIT)]
        return df[df[PARK_COL].isin(selected_parks)].copy().reset_index(drop=True)

    return df.copy().reset_index(drop=True)


def summarize_split(df, split_name):
    y = df[TARGET_COL]
    return {
        "split": split_name,
        "rows": len(df),
        "parks": df[PARK_COL].nunique(dropna=True),
        "min_timestamp": df[TIME_COL].min(),
        "max_timestamp": df[TIME_COL].max(),
        "target_mean": y.mean(),
        "target_std": y.std(),
        "target_min": y.min(),
        "target_max": y.max(),
    }


train_raw_df = load_canonical_split(TRAIN_PATH, "train")
val_raw_df = load_canonical_split(VAL_PATH, "validation")
test_raw_df = load_canonical_split(TEST_PATH, "test")

train_df = apply_working_subset(train_raw_df)
val_df = apply_working_subset(val_raw_df)
test_df = apply_working_subset(test_raw_df)

split_summary_df = pd.DataFrame([
    summarize_split(train_df, "train"),
    summarize_split(val_df, "validation"),
    summarize_split(test_df, "test"),
])

display(split_summary_df)


## Schema And Feature Audit

Το feature audit γίνεται με βάση το train split. Τα features για το fallback diagnostic model είναι numeric και αποκλείουν identifiers, timestamps, target, test flags, prediction columns και προφανείς leakage columns. Τα metadata και regime columns παραμένουν διαθέσιμα μόνο για residual interpretation.


In [ ]:
# ============================================================
# NB20 | Feature schema audit για controlled fallback model
# ============================================================

EXPLICIT_EXCLUDE_COLUMNS = {
    TARGET_COL,
    PARK_COL,
    TIME_COL,
    TEST_FLAG_COL,
    "split",
    *PREDICTION_COLUMN_CANDIDATES,
    "y_true",
    "y_pred",
    "residual",
    "abs_error",
    "squared_error",
}

LEAKAGE_NAME_PATTERNS = [
    "future",
    "lead_",
    "_lead",
    "residual",
    "error",
    "forecast_error",
]


def is_leakage_like_column(column_name):
    lower = column_name.lower()
    return any(pattern in lower for pattern in LEAKAGE_NAME_PATTERNS)


numeric_train_columns = train_df.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [
    col for col in numeric_train_columns
    if col not in EXPLICIT_EXCLUDE_COLUMNS and not is_leakage_like_column(col)
]

available_metadata_cols = [col for col in METADATA_COLUMNS if col in train_df.columns or col in test_df.columns]
available_regime_cols = [col for col in REGIME_COLUMNS if col in train_df.columns or col in test_df.columns]
available_lagged_wind_cols = [
    col for col in train_df.columns
    if ("lag" in col.lower() or "_m" in col.lower() or "_p" in col.lower())
    and any(token in col for token in ["U_GVL", "V_GVL", "Wind_Speed", "wind"])
]

feature_audit_df = pd.DataFrame({
    "audit_item": [
        "numeric_train_columns",
        "diagnostic_fallback_feature_count",
        "available_metadata_columns",
        "available_regime_columns",
        "available_lagged_wind_vector_columns",
    ],
    "value": [
        len(numeric_train_columns),
        len(feature_cols),
        ", ".join(available_metadata_cols),
        ", ".join(available_regime_cols),
        ", ".join(available_lagged_wind_cols[:30]),
    ],
})

display(feature_audit_df)
display(pd.DataFrame({"feature_column": feature_cols}).head(60))

if len(feature_cols) == 0:
    raise ValueError("No numeric fallback features available after leakage exclusions.")


## Prediction Source Resolver

Η resolver πολιτική προτιμά existing prediction columns στα validation και test canonical splits. Αν λείπουν validation predictions, το notebook μπορεί να εκπαιδεύσει μόνο train-based fallback diagnostic model και να χρησιμοποιήσει validation μόνο για calibration/thresholds. Τα optional test prediction CSVs χρησιμοποιούνται μόνο για model-comparison residual interpretation.


In [ ]:
# ============================================================
# NB20 | Prediction source audit ??? resolver
# ============================================================

PREDICTION_PROVENANCE_NOTE = (
    "NB20 treats canonical split prediction columns as upstream canonical predictions; "
    "it does not retrain, overwrite, or alter canonical benchmark files."
)


def prediction_column_quality(df, column):
    present = column in df.columns
    if not present:
        return {
            "present": False,
            "non_null_predictions": 0,
            "usable": False,
            "constant_prediction_check": False,
            "prediction_equals_target_rate": np.nan,
            "prediction_equals_target_suspicious_check": False,
            "rejection_reasons": "missing_column",
        }

    values = pd.to_numeric(df[column], errors="coerce")
    target = pd.to_numeric(df[TARGET_COL], errors="coerce") if TARGET_COL in df.columns else pd.Series(np.nan, index=df.index)
    usable_mask = values.notna() & target.notna()
    non_null_predictions = int(values.notna().sum())
    unique_predictions = int(values.dropna().nunique())
    constant_prediction_check = bool(non_null_predictions > 0 and unique_predictions <= 1)
    prediction_equals_target_rate = float(np.isclose(values[usable_mask], target[usable_mask], atol=1e-12, rtol=0.0).mean()) if usable_mask.any() else np.nan
    prediction_equals_target_suspicious_check = bool(pd.notna(prediction_equals_target_rate) and prediction_equals_target_rate > 0.99)

    rejection_reasons = []
    if non_null_predictions == 0:
        rejection_reasons.append("no_non_null_predictions")
    if constant_prediction_check:
        rejection_reasons.append("constant_prediction")
    if prediction_equals_target_suspicious_check:
        rejection_reasons.append("prediction_equals_target_rate_gt_0_99")

    return {
        "present": True,
        "non_null_predictions": non_null_predictions,
        "usable": len(rejection_reasons) == 0,
        "constant_prediction_check": constant_prediction_check,
        "prediction_equals_target_rate": prediction_equals_target_rate,
        "prediction_equals_target_suspicious_check": prediction_equals_target_suspicious_check,
        "rejection_reasons": ";".join(rejection_reasons) if rejection_reasons else "none",
    }


def usable_prediction_column(df, column):
    return bool(prediction_column_quality(df, column)["usable"])


def canonical_prediction_audit(split_name, df):
    rows = []
    for column in PREDICTION_COLUMN_CANDIDATES:
        quality = prediction_column_quality(df, column)
        present = quality["present"]
        rows.append({
            "source_group": "canonical_split_column",
            "source_name": split_name,
            "column": column,
            "path": f"{split_name}_final.csv",
            "exists": present,
            "n_rows_checked": len(df),
            "non_null_predictions": quality["non_null_predictions"],
            "usable": quality["usable"],
            "constant_prediction_check": quality["constant_prediction_check"],
            "prediction_equals_target_rate": quality["prediction_equals_target_rate"],
            "prediction_equals_target_suspicious_check": quality["prediction_equals_target_suspicious_check"],
            "rejection_reasons": quality["rejection_reasons"],
            "prediction_provenance_status": "upstream_canonical_prediction_column" if present else "not_available",
            "prediction_provenance_note": PREDICTION_PROVENANCE_NOTE if present else "Column not present in this canonical split.",
            "schema_columns": ", ".join(df.columns[:12]),
        })
    return rows


def optional_prediction_audit(source_name, path):
    exists = path.exists()
    schema_columns = []
    n_preview_rows = 0
    usable = False
    if exists:
        try:
            preview = pd.read_csv(path, nrows=25)
            schema_columns = list(preview.columns)
            n_preview_rows = len(preview)
            usable = bool({"park_id", "timestamp"}.issubset(preview.columns)) and any(
                col in preview.columns for col in ["y_pred", "prediction", "predicted", "model_prediction"]
            )
        except Exception as exc:
            schema_columns = [f"schema_error: {exc}"]
    return {
        "source_group": "optional_prediction_csv",
        "source_name": source_name,
        "column": "inferred_at_load_time",
        "path": str(path.relative_to(PROJECT_ROOT)) if exists else str(path),
        "exists": exists,
        "n_rows_checked": n_preview_rows,
        "non_null_predictions": np.nan,
        "usable": usable,
        "constant_prediction_check": np.nan,
        "prediction_equals_target_rate": np.nan,
        "prediction_equals_target_suspicious_check": np.nan,
        "rejection_reasons": "not_used_for_threshold_resolution",
        "prediction_provenance_status": "optional_test_prediction_csv" if exists else "not_available",
        "prediction_provenance_note": "Optional prediction CSVs are used only for model-comparison residual interpretation, not threshold derivation.",
        "schema_columns": ", ".join(schema_columns[:20]),
    }


prediction_source_audit_rows = []
prediction_source_audit_rows.extend(canonical_prediction_audit("train", train_df))
prediction_source_audit_rows.extend(canonical_prediction_audit("validation", val_df))
prediction_source_audit_rows.extend(canonical_prediction_audit("test", test_df))
for source_name, path in OPTIONAL_PREDICTION_PATHS.items():
    prediction_source_audit_rows.append(optional_prediction_audit(source_name, path))

prediction_source_audit_df = pd.DataFrame(prediction_source_audit_rows)


def resolve_split_prediction_source(val_frame, test_frame):
    direct_rejection_records = []
    for column in PREDICTION_COLUMN_CANDIDATES:
        val_quality = prediction_column_quality(val_frame, column)
        test_quality = prediction_column_quality(test_frame, column)
        if val_quality["usable"] and test_quality["usable"]:
            return {
                "kind": "canonical_split_column",
                "label": f"canonical_split_column:{column}",
                "prediction_column": column,
                "requires_fallback_model": False,
                "validation_threshold_source": "validation canonical split predictions",
                "selected_prediction_not_constant": True,
                "selected_prediction_not_equal_target": True,
                "prediction_provenance_status": "upstream_canonical_prediction_column",
                "prediction_provenance_note": PREDICTION_PROVENANCE_NOTE,
                "direct_rejection_summary": "none",
            }
        direct_rejection_records.append(
            f"{column}:validation={val_quality['rejection_reasons']};test={test_quality['rejection_reasons']}"
        )
    return {
        "kind": "controlled_diagnostic_fallback_model",
        "label": "controlled_diagnostic_fallback_model",
        "prediction_column": None,
        "requires_fallback_model": True,
        "validation_threshold_source": "validation predictions from train-only fallback model",
        "selected_prediction_not_constant": True,
        "selected_prediction_not_equal_target": True,
        "prediction_provenance_status": "fallback_train_only_diagnostic_model",
        "prediction_provenance_note": "All direct canonical prediction columns failed validation/test sanity checks; NB20 trains a diagnostic fallback only for leakage-safe residual generation.",
        "direct_rejection_summary": " | ".join(direct_rejection_records),
    }


selected_prediction_source = resolve_split_prediction_source(val_df, test_df)
prediction_source_resolved = bool(selected_prediction_source["label"])

prediction_source_audit_df["selected_by_resolver"] = (
    (prediction_source_audit_df["source_group"] == "canonical_split_column")
    & (prediction_source_audit_df["column"] == selected_prediction_source.get("prediction_column"))
    & (prediction_source_audit_df["source_name"].isin(["validation", "test"]))
)

fallback_resolution_audit_df = pd.DataFrame([{
    "selected_prediction_source": selected_prediction_source["label"],
    "requires_fallback_model": selected_prediction_source["requires_fallback_model"],
    "direct_rejection_summary": selected_prediction_source.get("direct_rejection_summary", "none"),
    "prediction_provenance_status": selected_prediction_source["prediction_provenance_status"],
    "prediction_provenance_note": selected_prediction_source["prediction_provenance_note"],
}])

display(prediction_source_audit_df)
display(pd.DataFrame([selected_prediction_source]))
display(fallback_resolution_audit_df)


## Controlled Fallback Diagnostic Model

Αυτή η cell εκτελείται μόνο αν δεν υπάρχουν usable validation και test prediction columns στα canonical splits. Το fallback model υπάρχει αποκλειστικά για leakage-safe residual generation και όχι ως νέο canonical benchmark.


In [ ]:
# ============================================================
# NB20 | Fallback model μόνο όταν λείπουν validation predictions
# ============================================================

fallback_model_audit_df = pd.DataFrame()
fallback_validation_metrics_df = pd.DataFrame()
fallback_selected_test_metrics_df = pd.DataFrame()
fallback_train_validation_test_audit_df = pd.DataFrame()
test_evaluation_count = 1

val_prediction_df = val_df.copy()
test_prediction_df = test_df.copy()


def regression_metrics(y_true, y_pred):
    y_true = pd.to_numeric(pd.Series(y_true), errors="coerce")
    y_pred = pd.to_numeric(pd.Series(y_pred), errors="coerce")
    mask = y_true.notna() & y_pred.notna()
    y_true = y_true[mask].to_numpy(dtype=float)
    y_pred = y_pred[mask].to_numpy(dtype=float)
    if len(y_true) == 0:
        return {"MAE": np.nan, "RMSE": np.nan, "R2": np.nan}
    residual = y_true - y_pred
    mae = float(np.mean(np.abs(residual)))
    rmse = float(np.sqrt(np.mean(residual ** 2)))
    ss_res = float(np.sum(residual ** 2))
    ss_tot = float(np.sum((y_true - np.mean(y_true)) ** 2))
    r2 = float(1.0 - ss_res / ss_tot) if len(y_true) > 1 and ss_tot > 0 else np.nan
    return {"MAE": mae, "RMSE": rmse, "R2": r2}


def prepare_xy(frame, columns):
    X = frame[columns].copy()
    y = pd.to_numeric(frame[TARGET_COL], errors="coerce")
    mask = y.notna()
    return X.loc[mask], y.loc[mask]


if not selected_prediction_source["requires_fallback_model"]:
    prediction_column = selected_prediction_source["prediction_column"]
    val_prediction_df["diagnostic_prediction"] = pd.to_numeric(val_prediction_df[prediction_column], errors="coerce")
    test_prediction_df["diagnostic_prediction"] = pd.to_numeric(test_prediction_df[prediction_column], errors="coerce")
    fallback_model_audit_df = pd.DataFrame([{
        "fallback_required": False,
        "selected_prediction_source": selected_prediction_source["label"],
        "prediction_provenance_status": selected_prediction_source["prediction_provenance_status"],
        "prediction_provenance_note": selected_prediction_source["prediction_provenance_note"],
        "note": "Existing canonical split predictions used; no fallback model trained.",
    }])
    fallback_train_validation_test_audit_df = pd.DataFrame([{
        "fallback_used": False,
        "selected_prediction_source": selected_prediction_source["label"],
        "diagnostic_only_note": "Fallback audit not applicable because upstream canonical predictions were selected.",
    }])
else:
    if not SKLEARN_AVAILABLE:
        raise ImportError(f"scikit-learn is required for the controlled fallback diagnostic model: {SKLEARN_IMPORT_ERROR}")
    X_train_raw, y_train = prepare_xy(train_df, feature_cols)
    X_val_raw, y_val = prepare_xy(val_df, feature_cols)
    X_test_raw, y_test = prepare_xy(test_df, feature_cols)

    imputer = SimpleImputer(strategy="median")
    X_train = imputer.fit_transform(X_train_raw)
    X_val = imputer.transform(X_val_raw)
    X_test = imputer.transform(X_test_raw)

    candidate_rows = []
    model_candidates = []

    try:
        from xgboost import XGBRegressor
        for config_id, params in enumerate(itertools.product([300, 500], [4, 6], [0.03, 0.05], [0.8], [0.8]), start=1):
            n_estimators, max_depth, learning_rate, subsample, colsample_bytree = params
            model = XGBRegressor(
                objective="reg:squarederror",
                random_state=SEED,
                n_estimators=n_estimators,
                max_depth=max_depth,
                learning_rate=learning_rate,
                subsample=subsample,
                colsample_bytree=colsample_bytree,
                tree_method="hist",
                n_jobs=-1,
                eval_metric="rmse",
            )
            model_candidates.append((
                config_id,
                "XGBoost diagnostic fallback",
                {
                    "n_estimators": n_estimators,
                    "max_depth": max_depth,
                    "learning_rate": learning_rate,
                    "subsample": subsample,
                    "colsample_bytree": colsample_bytree,
                },
                model,
            ))
    except Exception as exc:
        fallback_import_note = f"xgboost unavailable: {exc}"
        for config_id, params in enumerate(itertools.product([300, 500], [31, 63], [0.03, 0.05]), start=1):
            max_iter, max_leaf_nodes, learning_rate = params
            model = HistGradientBoostingRegressor(
                max_iter=max_iter,
                max_leaf_nodes=max_leaf_nodes,
                learning_rate=learning_rate,
                random_state=SEED,
            )
            model_candidates.append((
                config_id,
                "HistGradientBoosting diagnostic fallback",
                {
                    "max_iter": max_iter,
                    "max_leaf_nodes": max_leaf_nodes,
                    "learning_rate": learning_rate,
                    "import_note": fallback_import_note,
                },
                model,
            ))

    fitted_models = {}
    for config_id, model_name, params, model in model_candidates:
        model.fit(X_train, y_train)
        val_pred = model.predict(X_val)
        metrics = regression_metrics(y_val, val_pred)
        candidate_rows.append({
            "config_id": config_id,
            "model": model_name,
            **params,
            **metrics,
        })
        fitted_models[config_id] = model

    fallback_validation_metrics_df = pd.DataFrame(candidate_rows).sort_values(
        ["MAE", "RMSE", "R2"], ascending=[True, True, False]
    ).reset_index(drop=True)
    fallback_validation_metrics_df.insert(0, "validation_rank", np.arange(1, len(fallback_validation_metrics_df) + 1))

    selected_row = fallback_validation_metrics_df.iloc[0].to_dict()
    selected_model = fitted_models[int(selected_row["config_id"])]
    train_selected_pred = selected_model.predict(X_train)
    val_selected_pred = selected_model.predict(X_val)
    test_selected_pred = selected_model.predict(X_test)

    val_prediction_df = val_df.loc[y_val.index].copy()
    test_prediction_df = test_df.loc[y_test.index].copy()
    val_prediction_df["diagnostic_prediction"] = val_selected_pred
    test_prediction_df["diagnostic_prediction"] = test_selected_pred

    train_metrics = regression_metrics(y_train, train_selected_pred)
    validation_metrics = regression_metrics(y_val, val_selected_pred)
    test_metrics = regression_metrics(y_test, test_selected_pred)
    fallback_selected_test_metrics_df = pd.DataFrame([{
        "model": selected_row["model"],
        "config_id": int(selected_row["config_id"]),
        "test_evaluations": 1,
        **test_metrics,
    }])
    fallback_train_validation_test_audit_df = pd.DataFrame([{
        "fallback_used": True,
        "selected_prediction_source": f"fallback:{selected_row['model']}:config_{int(selected_row['config_id'])}",
        "model": selected_row["model"],
        "config_id": int(selected_row["config_id"]),
        "train_MAE": train_metrics["MAE"],
        "train_RMSE": train_metrics["RMSE"],
        "train_R2": train_metrics["R2"],
        "validation_MAE": validation_metrics["MAE"],
        "validation_RMSE": validation_metrics["RMSE"],
        "validation_R2": validation_metrics["R2"],
        "test_MAE": test_metrics["MAE"],
        "test_RMSE": test_metrics["RMSE"],
        "test_R2": test_metrics["R2"],
        "train_validation_gap_MAE": validation_metrics["MAE"] - train_metrics["MAE"],
        "validation_test_gap_MAE": test_metrics["MAE"] - validation_metrics["MAE"],
        "diagnostic_only_note": "Fallback model is diagnostic-only for residual generation and is not a canonical benchmark.",
    }])

    selected_prediction_source.update({
        "label": f"fallback:{selected_row['model']}:config_{int(selected_row['config_id'])}",
        "selected_config_id": int(selected_row["config_id"]),
        "selected_model": selected_row["model"],
        "prediction_provenance_status": "fallback_train_only_diagnostic_model",
        "prediction_provenance_note": "Fallback trained on train split only; validation is used for selection/calibration and test is evaluated once. Diagnostic only, not canonical benchmark.",
    })
    test_evaluation_count = 1

    fallback_model_audit_df = pd.DataFrame([{
        "fallback_required": True,
        "selected_prediction_source": selected_prediction_source["label"],
        "train_rows": len(X_train_raw),
        "validation_rows": len(X_val_raw),
        "test_rows": len(X_test_raw),
        "feature_count": len(feature_cols),
        "test_evaluations": test_evaluation_count,
        "prediction_provenance_status": selected_prediction_source["prediction_provenance_status"],
        "prediction_provenance_note": selected_prediction_source["prediction_provenance_note"],
    }])

display(fallback_model_audit_df)
if not fallback_validation_metrics_df.empty:
    display(fallback_validation_metrics_df)
if not fallback_selected_test_metrics_df.empty:
    display(fallback_selected_test_metrics_df)
if not fallback_train_validation_test_audit_df.empty:
    display(fallback_train_validation_test_audit_df)


## Run Mode Dashboard

Το dashboard συγκεντρώνει branch/commit context όταν είναι διαθέσιμο, τα active flags, τις βασικές εισόδους/εξόδους και το prediction-source state. Είναι audit surface για scaffold και full local runs.


In [ ]:
# ============================================================
# NB20 | Run mode dashboard
# ============================================================

def run_git_command(args):
    try:
        result = subprocess.run(
            ["git", *args],
            cwd=PROJECT_ROOT,
            capture_output=True,
            text=True,
            check=True,
        )
        return result.stdout.strip()
    except Exception:
        return "unavailable"


fallback_model_was_trained = bool(
    not fallback_model_audit_df.empty
    and bool(fallback_model_audit_df.get("fallback_required", pd.Series([False])).iloc[0])
)

run_mode_dashboard_df = pd.DataFrame([
    {"item": "current_branch", "value": run_git_command(["branch", "--show-current"])},
    {"item": "current_git_commit", "value": run_git_command(["rev-parse", "--short", "HEAD"])},
    {"item": "SMOKE_MODE", "value": SMOKE_MODE},
    {"item": "RUN_FULL_DIAGNOSTICS", "value": RUN_FULL_DIAGNOSTICS},
    {"item": "EXPORT_RESULTS", "value": EXPORT_RESULTS},
    {"item": "FULL_EXPORT_RESIDUAL_RECORDS", "value": FULL_EXPORT_RESIDUAL_RECORDS},
    {"item": "train_path", "value": str(TRAIN_PATH.relative_to(PROJECT_ROOT))},
    {"item": "validation_path", "value": str(VAL_PATH.relative_to(PROJECT_ROOT))},
    {"item": "test_path", "value": str(TEST_PATH.relative_to(PROJECT_ROOT))},
    {"item": "output_path", "value": str(OUTPUT_DIR.relative_to(PROJECT_ROOT))},
    {"item": "selected_prediction_source", "value": selected_prediction_source["label"]},
    {"item": "prediction_provenance_status", "value": selected_prediction_source.get("prediction_provenance_status", "missing")},
    {"item": "prediction_provenance_note", "value": selected_prediction_source.get("prediction_provenance_note", "missing")},
    {"item": "fallback_model_was_trained", "value": fallback_model_was_trained},
    {"item": "full_residual_export_enabled", "value": bool(EXPORT_RESULTS and FULL_EXPORT_RESIDUAL_RECORDS)},
    {"item": "figures_will_be_written", "value": bool(EXPORT_RESULTS)},
])

display(run_mode_dashboard_df)


## Residual Construction

Τα residual records δημιουργούνται για validation και test με κοινό contract: `y_true`, `y_pred`, `residual = y_true - y_pred`, absolute/squared error, sign, identifiers και διαθέσιμα metadata/regime fields.


In [ ]:
# ============================================================
# NB20 | Residual records για validation και test
# ============================================================

def infer_wind_speed(frame):
    '''Υπολογίζει διαθέσιμο wind-speed proxy για operating-regime interpretation.'''
    if "Wind_Speed_100m_ms" in frame.columns:
        return pd.to_numeric(frame["Wind_Speed_100m_ms"], errors="coerce")
    if "ws_ref" in frame.columns:
        return pd.to_numeric(frame["ws_ref"], errors="coerce")
    uv_pairs = [("U_GVL_60_HL", "V_GVL_60_HL"), ("U_GVL_58_HL", "V_GVL_58_HL")]
    for u_col, v_col in uv_pairs:
        if u_col in frame.columns and v_col in frame.columns:
            u = pd.to_numeric(frame[u_col], errors="coerce")
            v = pd.to_numeric(frame[v_col], errors="coerce")
            return np.sqrt(u ** 2 + v ** 2)
    return pd.Series(np.nan, index=frame.index)


def build_residual_records(frame, split_name, y_pred_col="diagnostic_prediction"):
    keep_cols = [PARK_COL, TIME_COL, TARGET_COL, y_pred_col]
    interpretation_cols = [
        col for col in [*METADATA_COLUMNS, *REGIME_COLUMNS]
        if col in frame.columns and col not in keep_cols
    ]
    residual_df = frame[keep_cols + interpretation_cols].copy()
    residual_df = residual_df.rename(columns={TARGET_COL: "y_true", y_pred_col: "y_pred"})
    residual_df["y_true"] = pd.to_numeric(residual_df["y_true"], errors="coerce")
    residual_df["y_pred"] = pd.to_numeric(residual_df["y_pred"], errors="coerce")
    residual_df = residual_df.dropna(subset=[PARK_COL, TIME_COL, "y_true", "y_pred"]).copy()
    residual_df["residual"] = residual_df["y_true"] - residual_df["y_pred"]
    residual_df["abs_error"] = residual_df["residual"].abs()
    residual_df["squared_error"] = residual_df["residual"] ** 2
    residual_df["residual_sign"] = np.select(
        [residual_df["residual"] > 0, residual_df["residual"] < 0],
        ["observed_above_prediction", "observed_below_prediction"],
        default="zero_residual",
    )
    residual_df["split"] = split_name
    residual_df["prediction_source"] = selected_prediction_source["label"]
    residual_df["wind_speed_ms"] = infer_wind_speed(residual_df)
    residual_df = residual_df.sort_values([PARK_COL, TIME_COL]).reset_index(drop=True)
    return residual_df


val_residual_df = build_residual_records(val_prediction_df, "validation")
test_residual_df = build_residual_records(test_prediction_df, "test")

residual_record_audit_df = pd.DataFrame([
    {"split": "validation", "rows": len(val_residual_df), "parks": val_residual_df[PARK_COL].nunique()},
    {"split": "test", "rows": len(test_residual_df), "parks": test_residual_df[PARK_COL].nunique()},
])

display(residual_record_audit_df)
display(test_residual_df.head())

if val_residual_df.empty or test_residual_df.empty:
    raise ValueError("Residual construction produced empty validation or test residual records.")


## Prediction Alignment And Sanity Checks

Οι παρακάτω έλεγχοι τεκμηριώνουν ότι το selected prediction source ευθυγραμμίζεται με το target και δεν εμφανίζει προφανή σημάδια constant ή suspicious identity prediction. Δεν γίνεται clipping στις προβλέψεις.


In [ ]:
# ============================================================
# NB20 | Prediction alignment and sanity checks
# ============================================================

def prediction_alignment_summary(frame, split_name):
    usable = frame[["y_true", "y_pred"]].notna().all(axis=1)
    y_true = pd.to_numeric(frame.loc[usable, "y_true"], errors="coerce")
    y_pred = pd.to_numeric(frame.loc[usable, "y_pred"], errors="coerce")
    prediction_equals_target_rate = float(np.isclose(y_true, y_pred, atol=1e-12, rtol=0.0).mean()) if len(y_true) else np.nan
    return {
        "split": split_name,
        "rows_with_usable_y_true_y_pred": int(usable.sum()),
        "null_prediction_count": int(frame["y_pred"].isna().sum()),
        "prediction_min": float(y_pred.min()) if len(y_pred) else np.nan,
        "prediction_max": float(y_pred.max()) if len(y_pred) else np.nan,
        "prediction_mean": float(y_pred.mean()) if len(y_pred) else np.nan,
        "target_min": float(y_true.min()) if len(y_true) else np.nan,
        "target_max": float(y_true.max()) if len(y_true) else np.nan,
        "target_mean": float(y_true.mean()) if len(y_true) else np.nan,
        "constant_prediction_check": bool(y_pred.nunique(dropna=True) <= 1) if len(y_pred) else False,
        "prediction_equals_target_suspicious_check": bool(prediction_equals_target_rate > 0.99) if len(y_true) else False,
        "prediction_equals_target_rate": prediction_equals_target_rate,
        "clipping_applied": False,
        "alignment_mode_used": selected_prediction_source["kind"],
    }


prediction_alignment_sanity_df = pd.DataFrame([
    prediction_alignment_summary(val_residual_df, "validation"),
    prediction_alignment_summary(test_residual_df, "test"),
])

selected_prediction_not_constant = not bool(prediction_alignment_sanity_df["constant_prediction_check"].any())
selected_prediction_not_equal_target = not bool(prediction_alignment_sanity_df["prediction_equals_target_suspicious_check"].any())

selected_prediction_source["selected_prediction_not_constant"] = selected_prediction_not_constant
selected_prediction_source["selected_prediction_not_equal_target"] = selected_prediction_not_equal_target

display(prediction_alignment_sanity_df)


## Validation-Derived Threshold Policy

Όλα τα residual thresholds παράγονται αποκλειστικά από validation residuals. Το test split χρησιμοποιείται μόνο μετά το κλείδωμα των thresholds.


In [ ]:
# ============================================================
# NB20 | Thresholds από validation residuals μόνο
# ============================================================

def safe_quantile(series, q):
    values = pd.to_numeric(series, errors="coerce").dropna()
    if values.empty:
        return np.nan
    return float(values.quantile(q))


def safe_mad(series):
    values = pd.to_numeric(series, errors="coerce").dropna()
    if values.empty:
        return np.nan
    median = values.median()
    return float((values - median).abs().median())


def threshold_stats_for_group(group, scope, park_id="__GLOBAL__"):
    residual = pd.to_numeric(group["residual"], errors="coerce")
    abs_error = pd.to_numeric(group["abs_error"], errors="coerce")
    return {
        "scope": scope,
        "park_id": park_id,
        "n_validation_rows": int(len(group)),
        "sufficient_validation_rows": bool(len(group) >= MIN_PARK_VALIDATION_ROWS),
        "residual_mean": float(residual.mean()),
        "residual_std": float(residual.std()),
        "residual_median": float(residual.median()),
        "residual_mad": safe_mad(residual),
        "abs_error_q90": safe_quantile(abs_error, 0.90),
        "abs_error_q95": safe_quantile(abs_error, 0.95),
        "abs_error_q99": safe_quantile(abs_error, 0.99),
    }


threshold_lookup_rows = [threshold_stats_for_group(val_residual_df, "global")]
for park_id, group in val_residual_df.groupby(PARK_COL, sort=True):
    threshold_lookup_rows.append(threshold_stats_for_group(group, "park", park_id=park_id))

threshold_lookup_df = pd.DataFrame(threshold_lookup_rows)


def threshold_lookup_to_policy(lookup_df, threshold_columns, source_split="validation"):
    rows = []
    for _, row in lookup_df.iterrows():
        for column in threshold_columns:
            rows.append({
                "scope": row["scope"],
                "park_id": row["park_id"],
                "threshold_name": column,
                "threshold_value": row.get(column, np.nan),
                "source_split": source_split,
                "n_validation_rows": row["n_validation_rows"],
                "fallback_used": bool(row["scope"] == "park" and not row["sufficient_validation_rows"]),
            })
    return pd.DataFrame(rows)


threshold_policy_df = threshold_lookup_to_policy(
    threshold_lookup_df,
    [
        "residual_mean",
        "residual_std",
        "residual_median",
        "residual_mad",
        "abs_error_q90",
        "abs_error_q95",
        "abs_error_q99",
    ],
)


def attach_residual_thresholds(frame, lookup_df):
    threshold_cols = [
        "residual_mean",
        "residual_std",
        "residual_median",
        "residual_mad",
        "abs_error_q90",
        "abs_error_q95",
        "abs_error_q99",
    ]
    global_row = lookup_df.query("scope == 'global'").iloc[0]
    park_lookup = lookup_df.query("scope == 'park' and sufficient_validation_rows == True")[[PARK_COL, *threshold_cols]].copy()
    merged = frame.merge(park_lookup, on=PARK_COL, how="left")
    has_park_threshold = merged["abs_error_q95"].notna()
    for column in threshold_cols:
        merged[column] = merged[column].fillna(global_row[column])
    denominator = 1.4826 * merged["residual_mad"].replace(0, np.nan)
    denominator = denominator.fillna(merged["residual_std"].replace(0, np.nan))
    denominator = denominator.fillna(global_row["residual_std"] if pd.notna(global_row["residual_std"]) else 1.0)
    denominator = denominator.replace(0, 1.0)
    standard_denominator = merged["residual_std"].abs().where(merged["residual_std"].abs() > 1e-12, np.nan)
    standard_denominator = standard_denominator.fillna(global_row["residual_std"] if pd.notna(global_row["residual_std"]) else 1.0)
    standard_denominator = standard_denominator.abs().where(standard_denominator.abs() > 1e-12, 1.0)
    merged["residual_z_score"] = (merged["residual"] - merged["residual_mean"]) / standard_denominator
    merged["robust_z"] = (merged["residual"] - merged["residual_median"]) / denominator
    merged["robust_residual_z_score"] = merged["robust_z"]
    merged["threshold_scope"] = np.where(has_park_threshold, "park", "global_fallback")
    return merged


val_residual_df = attach_residual_thresholds(val_residual_df, threshold_lookup_df)
test_residual_df = attach_residual_thresholds(test_residual_df, threshold_lookup_df)

validation_thresholds_used = True
no_test_threshold_leakage = True

display(threshold_policy_df.head(30))


## Rolling Residual Diagnostics

Τα rolling indicators υπολογίζονται ανά park και timestamp. Τα rolling thresholds παράγονται ξανά μόνο από validation rolling residuals.


In [ ]:
# ============================================================
# NB20 | Rolling MAE, bias, RMSE και warning flags
# ============================================================

def add_rolling_diagnostics(frame):
    out = frame.sort_values([PARK_COL, TIME_COL]).copy()
    grouped = out.groupby(PARK_COL, group_keys=False)
    for window in ROLLING_WINDOWS:
        min_periods = ROLLING_MIN_PERIODS.get(window, max(2, window // 4))
        out[f"rolling_MAE_{window}"] = grouped["abs_error"].transform(
            lambda s: s.rolling(window=window, min_periods=min_periods).mean()
        )
        out[f"rolling_bias_{window}"] = grouped["residual"].transform(
            lambda s: s.rolling(window=window, min_periods=min_periods).mean()
        )
        out[f"rolling_RMSE_{window}"] = grouped["squared_error"].transform(
            lambda s: np.sqrt(s.rolling(window=window, min_periods=min_periods).mean())
        )
    return out.reset_index(drop=True)


val_residual_df = add_rolling_diagnostics(val_residual_df)
test_residual_df = add_rolling_diagnostics(test_residual_df)


def rolling_threshold_stats(group, scope, park_id="__GLOBAL__"):
    return {
        "scope": scope,
        "park_id": park_id,
        "n_validation_rows": int(len(group)),
        "sufficient_validation_rows": bool(len(group) >= MIN_PARK_VALIDATION_ROWS),
        "rolling_MAE_24_q95": safe_quantile(group["rolling_MAE_24"], 0.95),
    }


rolling_lookup_rows = [rolling_threshold_stats(val_residual_df, "global")]
for park_id, group in val_residual_df.groupby(PARK_COL, sort=True):
    rolling_lookup_rows.append(rolling_threshold_stats(group, "park", park_id=park_id))
rolling_threshold_lookup_df = pd.DataFrame(rolling_lookup_rows)

threshold_policy_df = pd.concat([
    threshold_policy_df,
    threshold_lookup_to_policy(rolling_threshold_lookup_df, ["rolling_MAE_24_q95"]),
], ignore_index=True)


def attach_rolling_thresholds(frame, lookup_df):
    global_row = lookup_df.query("scope == 'global'").iloc[0]
    park_lookup = lookup_df.query("scope == 'park' and sufficient_validation_rows == True")[[PARK_COL, "rolling_MAE_24_q95"]]
    out = frame.merge(park_lookup, on=PARK_COL, how="left", suffixes=("", "_park"))
    out["rolling_MAE_24_q95"] = out["rolling_MAE_24_q95"].fillna(global_row["rolling_MAE_24_q95"])
    return out


def add_warning_flags(frame):
    out = frame.copy()
    out["warning_abs_q95"] = out["abs_error"] > out["abs_error_q95"]
    out["warning_robust_z"] = out["robust_z"].abs() > 3.0
    out["warning_rolling_mae_24"] = out["rolling_MAE_24"] > out["rolling_MAE_24_q95"]
    out["warning_any"] = out[["warning_abs_q95", "warning_robust_z", "warning_rolling_mae_24"]].any(axis=1)
    out["warning_flag"] = out["warning_any"]

    def reasons(row):
        labels = []
        if row["warning_abs_q95"]:
            labels.append("abs_error_gt_validation_q95")
        if row["warning_robust_z"]:
            labels.append("abs_robust_z_gt_3")
        if row["warning_rolling_mae_24"]:
            labels.append("rolling_MAE_24_gt_validation_q95")
        return ";".join(labels) if labels else "none"

    out["warning_reason"] = out.apply(reasons, axis=1)
    return out


val_residual_df = attach_rolling_thresholds(val_residual_df, rolling_threshold_lookup_df)
test_residual_df = attach_rolling_thresholds(test_residual_df, rolling_threshold_lookup_df)
val_residual_df = add_warning_flags(val_residual_df)
test_residual_df = add_warning_flags(test_residual_df)

display(threshold_policy_df.tail(20))
display(test_residual_df[[PARK_COL, TIME_COL, "abs_error", "residual_z_score", "robust_residual_z_score", "rolling_MAE_24", "warning_flag", "warning_reason"]].head())


## Residual Column Contract Audit

Η ακόλουθη audit cell επιβεβαιώνει ότι τα validation/test residual records έχουν το manuscript-facing residual schema, μαζί με rolling indicators και warning flags.


In [ ]:
# ============================================================
# NB20 | Residual column contract audit
# ============================================================

def residual_contract_audit_column(column):
    val_present = column in val_residual_df.columns
    test_present = column in test_residual_df.columns
    return {
        "required_residual_column": column,
        "present_in_validation_residuals": val_present,
        "present_in_test_residuals": test_present,
        "validation_dtype": str(val_residual_df[column].dtype) if val_present else "missing",
        "test_dtype": str(test_residual_df[column].dtype) if test_present else "missing",
        "validation_non_null_count": int(val_residual_df[column].notna().sum()) if val_present else 0,
        "test_non_null_count": int(test_residual_df[column].notna().sum()) if test_present else 0,
        "validation_missing_count": int(val_residual_df[column].isna().sum()) if val_present else len(val_residual_df),
        "test_missing_count": int(test_residual_df[column].isna().sum()) if test_present else len(test_residual_df),
    }


residual_column_contract_audit_df = pd.DataFrame([
    residual_contract_audit_column(column)
    for column in REQUIRED_RESIDUAL_CONTRACT_COLUMNS
])

display(residual_column_contract_audit_df)


## Threshold Calibration Sanity Checks

Η calibration audit συγκρίνει τις αναμενόμενες validation exceedance rates με τις πραγματικές validation/test rates. Αυτό δείχνει ότι οι warning rules κλειδώνονται στο validation και μεταφέρονται στο test.


In [ ]:
# ============================================================
# NB20 | Threshold calibration sanity checks
# ============================================================

global_threshold_row = threshold_lookup_df.query("scope == 'global'").iloc[0]

calibration_rows = []
for quantile, column, expected_exceedance in [
    (0.90, "abs_error_q90", 0.10),
    (0.95, "abs_error_q95", 0.05),
    (0.99, "abs_error_q99", 0.01),
]:
    threshold_value = float(global_threshold_row[column])
    calibration_rows.append({
        "calibration_rule": f"abs_error_gt_global_validation_q{int(quantile * 100)}",
        "threshold_value": threshold_value,
        "expected_validation_exceedance_rate": expected_exceedance,
        "actual_validation_exceedance_rate": float((val_residual_df["abs_error"] > threshold_value).mean()),
        "actual_test_exceedance_rate": float((test_residual_df["abs_error"] > threshold_value).mean()),
    })

threshold_calibration_sanity_df = pd.DataFrame(calibration_rows)

warning_rate_transfer_df = pd.DataFrame([
    {
        "warning_family": "robust_z_abs_gt_3",
        "validation_rate": float(val_residual_df["warning_robust_z"].mean()),
        "test_rate": float(test_residual_df["warning_robust_z"].mean()),
    },
    {
        "warning_family": "rolling_MAE_24_gt_validation_q95",
        "validation_rate": float(val_residual_df["warning_rolling_mae_24"].mean()),
        "test_rate": float(test_residual_df["warning_rolling_mae_24"].mean()),
    },
    {
        "warning_family": "combined_warning_flag",
        "validation_rate": float(val_residual_df["warning_flag"].mean()),
        "test_rate": float(test_residual_df["warning_flag"].mean()),
    },
])

display(threshold_calibration_sanity_df)
display(warning_rate_transfer_df)


## Warning-Event Extraction

Τα warning events είναι contiguous warning rows ανά park. Κάθε event περιγράφει επίμονη απόκλιση από expected forecast behavior και όχι labeled fault.


In [ ]:
# ============================================================
# NB20 | Extraction contiguous warning events ??? park
# ============================================================

WARNING_FLAG_COLUMNS = ["warning_abs_q95", "warning_robust_z", "warning_rolling_mae_24"]
WARNING_REASON_LABELS = {
    "warning_abs_q95": "abs_error_gt_validation_q95",
    "warning_robust_z": "abs_robust_z_gt_3",
    "warning_rolling_mae_24": "rolling_MAE_24_gt_validation_q95",
}


def dominant_warning_reason(group):
    counts = {WARNING_REASON_LABELS[col]: int(group[col].sum()) for col in WARNING_FLAG_COLUMNS}
    if not counts or max(counts.values()) == 0:
        return "none"
    return max(counts.items(), key=lambda item: item[1])[0]


def mode_or_nan(series):
    values = series.dropna()
    if values.empty:
        return np.nan
    return values.mode().iloc[0]


def infer_common_timestamp_interval(timestamps):
    ordered = pd.to_datetime(timestamps, errors="coerce").dropna().sort_values()
    diffs = ordered.diff().dropna()
    diffs = diffs[diffs > pd.Timedelta(0)]
    if diffs.empty:
        return pd.NaT
    mode_values = diffs.mode()
    return mode_values.iloc[0] if not mode_values.empty else diffs.median()


def summarize_warning_event(event_group, event_counter, park_id, inferred_interval, timestamp_gap_breaks_used):
    event_start = event_group[TIME_COL].min()
    event_end = event_group[TIME_COL].max()
    duration_hours_if_inferable = np.nan
    if pd.notna(inferred_interval):
        duration_hours_if_inferable = float(len(event_group) * inferred_interval.total_seconds() / 3600.0)
    row = {
        "event_id": f"E{event_counter:05d}",
        "park_id": park_id,
        "event_start": event_start,
        "event_end": event_end,
        "duration_rows": int(len(event_group)),
        "duration_hours_if_inferable": duration_hours_if_inferable,
        "timestamp_gap_breaks_used": bool(timestamp_gap_breaks_used),
        "inferred_timestamp_interval_hours": float(inferred_interval.total_seconds() / 3600.0) if pd.notna(inferred_interval) else np.nan,
        "max_abs_error": float(event_group["abs_error"].max()),
        "mean_abs_error": float(event_group["abs_error"].mean()),
        "max_rolling_MAE_24": float(event_group["rolling_MAE_24"].max()),
        "mean_residual": float(event_group["residual"].mean()),
        "dominant_warning_reason": dominant_warning_reason(event_group),
        "target_mean_during_event": float(event_group["y_true"].mean()),
        "prediction_mean_during_event": float(event_group["y_pred"].mean()),
    }
    if "wind_speed_ms" in event_group.columns:
        row["mean_wind_speed_ms"] = float(event_group["wind_speed_ms"].mean())
    if "T_HAG_2_M" in event_group.columns:
        row["mean_temperature"] = float(pd.to_numeric(event_group["T_HAG_2_M"], errors="coerce").mean())
    if "nwp_fcst_horiz_hours" in event_group.columns:
        row["mode_nwp_fcst_horiz_hours"] = mode_or_nan(pd.to_numeric(event_group["nwp_fcst_horiz_hours"], errors="coerce"))
    return row


def close_active_event(events, active_event_rows, event_counter, park_id, inferred_interval, timestamp_gap_breaks_used):
    if not active_event_rows:
        return event_counter
    event_counter += 1
    events.append(summarize_warning_event(
        pd.DataFrame(active_event_rows),
        event_counter,
        park_id,
        inferred_interval,
        timestamp_gap_breaks_used,
    ))
    return event_counter


def extract_warning_events(frame):
    events = []
    interval_rows = []
    event_counter = 0
    for park_id, group in frame.sort_values([PARK_COL, TIME_COL]).groupby(PARK_COL, sort=True):
        group = group.reset_index(drop=True)
        inferred_interval = infer_common_timestamp_interval(group[TIME_COL])
        timestamp_gap_breaks_used = pd.notna(inferred_interval)
        gap_tolerance = inferred_interval * 1.5 if pd.notna(inferred_interval) else pd.NaT
        interval_rows.append({
            "park_id": park_id,
            "inferred_timestamp_interval": inferred_interval,
            "gap_tolerance": gap_tolerance,
            "timestamp_gap_breaks_used": bool(timestamp_gap_breaks_used),
        })
        active_event_rows = []
        previous_timestamp = pd.NaT
        for _, row in group.iterrows():
            current_timestamp = row[TIME_COL]
            gap_break = False
            if active_event_rows and pd.notna(previous_timestamp) and pd.notna(gap_tolerance):
                gap_break = bool((current_timestamp - previous_timestamp) > gap_tolerance)
            if bool(row["warning_any"]):
                if gap_break:
                    event_counter = close_active_event(events, active_event_rows, event_counter, park_id, inferred_interval, timestamp_gap_breaks_used)
                    active_event_rows = []
                active_event_rows.append(row)
            elif active_event_rows:
                event_counter = close_active_event(events, active_event_rows, event_counter, park_id, inferred_interval, timestamp_gap_breaks_used)
                active_event_rows = []
            previous_timestamp = current_timestamp
        if active_event_rows:
            event_counter = close_active_event(events, active_event_rows, event_counter, park_id, inferred_interval, timestamp_gap_breaks_used)
    return pd.DataFrame(events), pd.DataFrame(interval_rows)


warning_event_summary_df, warning_event_interval_audit_df = extract_warning_events(test_residual_df)
display(warning_event_interval_audit_df)
display(warning_event_summary_df.head(20))


## Warning Event Severity Ranking

Τα warning events ταξινομούνται με conservative severity score ώστε να εντοπίζονται candidate diagnostic events. Δεν αποτελούν confirmed faults.


In [ ]:
# ============================================================
# NB20 | Warning event severity ranking
# ============================================================

if warning_event_summary_df.empty:
    top_warning_events_df = pd.DataFrame(columns=[
        "event_id",
        "park_id",
        "event_start",
        "event_end",
        "duration_rows",
        "max_abs_error",
        "mean_abs_error",
        "max_rolling_MAE_24",
        "mean_residual",
        "severity_score",
        "dominant_warning_reason",
    ])
else:
    warning_event_summary_df = warning_event_summary_df.copy()
    warning_event_summary_df["severity_score"] = (
        warning_event_summary_df["duration_rows"]
        * warning_event_summary_df["mean_abs_error"]
        * (1.0 + warning_event_summary_df["mean_residual"].abs())
    )
    top_warning_events_df = warning_event_summary_df.sort_values(
        ["severity_score", "duration_rows", "max_abs_error", "max_rolling_MAE_24"],
        ascending=[False, False, False, False],
    ).head(10).reset_index(drop=True)

display(top_warning_events_df)


## Park-Level Diagnostic Summary

Το park-level summary ταξινομεί τα test parks με βάση residual stress: MAE, warning rate και absolute mean residual. Η κατάταξη είναι condition-monitoring-oriented evidence και όχι επιβεβαίωση βλάβης.


In [ ]:
# ============================================================
# NB20 | Park-level residual diagnostics και ranking
# ============================================================

def summarize_residual_group(group):
    metrics = regression_metrics(group["y_true"], group["y_pred"])
    return {
        "n_test_rows": int(len(group)),
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "R2": metrics["R2"],
        "mean_residual": float(group["residual"].mean()),
        "median_residual": float(group["residual"].median()),
        "median_abs_error": float(group["abs_error"].median()),
        "p95_abs_error": safe_quantile(group["abs_error"], 0.95),
        "residual_std": float(group["residual"].std()),
        "warning_count": int(group["warning_any"].sum()),
        "warning_rate": float(group["warning_any"].mean()),
        "mean_target": float(group["y_true"].mean()),
        "mean_prediction": float(group["y_pred"].mean()),
    }


park_rows = []
for park_id, group in test_residual_df.groupby(PARK_COL, sort=True):
    row = {"park_id": park_id, **summarize_residual_group(group)}
    for column in METADATA_COLUMNS:
        if column in group.columns:
            values = pd.to_numeric(group[column], errors="coerce").dropna()
            row[column] = values.iloc[0] if not values.empty else np.nan
    park_rows.append(row)

park_level_summary_df = pd.DataFrame(park_rows)

if warning_event_summary_df.empty:
    event_counts_df = pd.DataFrame(columns=["park_id", "event_count", "longest_event_duration"])
else:
    event_counts_df = warning_event_summary_df.groupby("park_id").agg(
        event_count=("event_id", "count"),
        longest_event_duration=("duration_rows", "max"),
    ).reset_index()

park_level_summary_df = park_level_summary_df.merge(event_counts_df, on="park_id", how="left")
park_level_summary_df["event_count"] = park_level_summary_df["event_count"].fillna(0).astype(int)
park_level_summary_df["longest_event_duration"] = park_level_summary_df["longest_event_duration"].fillna(0).astype(int)
park_level_summary_df["abs_mean_residual"] = park_level_summary_df["mean_residual"].abs()
park_level_summary_df = park_level_summary_df.sort_values(
    ["MAE", "warning_rate", "abs_mean_residual"], ascending=[False, False, False]
).reset_index(drop=True)
park_level_summary_df.insert(0, "diagnostic_rank", np.arange(1, len(park_level_summary_df) + 1))

top_diagnostic_parks_df = park_level_summary_df.head(10).copy()

display(park_level_summary_df.head(20))
display(top_diagnostic_parks_df)


## Directional Bias Diagnostics

Η directional ανάλυση διαχωρίζει underprediction, overprediction και near-zero residual behavior. Αυτό βοηθά τη manuscript ερμηνεία να ξεχωρίζει συστηματική bias από απλή αύξηση absolute error.


In [ ]:
# ============================================================
# NB20 | Directional bias diagnostics
# ============================================================

NEAR_ZERO_RESIDUAL_TOLERANCE = 0.01


def directional_bias_metrics(group, split_name, scope, park_id="__ALL__"):
    residual = pd.to_numeric(group["residual"], errors="coerce")
    return {
        "split": split_name,
        "scope": scope,
        "park_id": park_id,
        "n_rows": int(len(group)),
        "underprediction_rate": float((residual > 0).mean()),
        "overprediction_rate": float((residual < 0).mean()),
        "near_zero_residual_rate": float((residual.abs() <= NEAR_ZERO_RESIDUAL_TOLERANCE).mean()),
        "mean_residual": float(residual.mean()),
        "median_residual": float(residual.median()),
        "warning_rate": float(group["warning_flag"].mean()),
    }


directional_rows = []
for split_name, frame in [("validation", val_residual_df), ("test", test_residual_df)]:
    directional_rows.append(directional_bias_metrics(frame, split_name, "overall"))
    for park_id, group in frame.groupby(PARK_COL, sort=True):
        directional_rows.append(directional_bias_metrics(group, split_name, "park", park_id=park_id))

directional_bias_summary_df = pd.DataFrame(directional_rows)

display(directional_bias_summary_df)


## Residual Persistence / Autocorrelation Diagnostics

Persistent residuals are more PHM-relevant than isolated spikes. This section measures autocorrelation and longest same-sign/warning runs per test park.


In [ ]:
# ============================================================
# NB20 | Residual persistence and autocorrelation diagnostics
# ============================================================

def lag_autocorrelation(series, lag):
    values = pd.to_numeric(series, errors="coerce").dropna()
    if len(values) <= lag + 2:
        return np.nan
    return float(values.autocorr(lag=lag))


def longest_true_run(values):
    longest = 0
    current = 0
    for value in values:
        if bool(value):
            current += 1
            longest = max(longest, current)
        else:
            current = 0
    return int(longest)


def longest_same_sign_run(residual):
    signs = np.sign(pd.to_numeric(residual, errors="coerce").fillna(0).to_numpy())
    longest = 0
    current = 0
    previous = 0
    for sign in signs:
        if sign == 0:
            current = 0
            previous = 0
        elif sign == previous:
            current += 1
        else:
            current = 1
            previous = sign
        longest = max(longest, current)
    return int(longest)


event_count_lookup = (
    warning_event_summary_df.groupby("park_id")["event_id"].count().to_dict()
    if not warning_event_summary_df.empty
    else {}
)

persistence_rows = []
for park_id, group in test_residual_df.sort_values([PARK_COL, TIME_COL]).groupby(PARK_COL, sort=True):
    persistence_rows.append({
        "park_id": park_id,
        "n_test_rows": int(len(group)),
        "lag_1_residual_autocorrelation": lag_autocorrelation(group["residual"], 1),
        "lag_24_residual_autocorrelation": lag_autocorrelation(group["residual"], 24),
        "lag_72_residual_autocorrelation": lag_autocorrelation(group["residual"], 72),
        "longest_same_sign_residual_run": longest_same_sign_run(group["residual"]),
        "longest_warning_run": longest_true_run(group["warning_flag"].to_numpy()),
        "warning_event_count": int(event_count_lookup.get(park_id, 0)),
        "warning_rate": float(group["warning_flag"].mean()),
    })

residual_persistence_summary_df = pd.DataFrame(persistence_rows)
display(residual_persistence_summary_df)


## Operating-Regime Residual Analysis

Η regime ανάλυση συγκεντρώνει residual stress σε target/predicted output bins, forecast horizon, wind-speed και temperature regimes όταν τα αντίστοιχα features υπάρχουν. Η ερμηνεία επιτρέπεται να πει ότι το residual stress συγκεντρώνεται σε high-output ή transition operating regimes, χωρίς causal claims.


In [ ]:
# ============================================================
# NB20 | Operating-regime bins και residual summaries
# ============================================================

POWER_BINS = [0.0, 0.10, 0.25, 0.50, 0.75, 0.90, 1.0]
POWER_BIN_LABELS = ["0.00-0.10", "0.10-0.25", "0.25-0.50", "0.50-0.75", "0.75-0.90", "0.90-1.00"]


def qbin_with_fallback(series, q=4):
    values = pd.to_numeric(series, errors="coerce")
    if values.notna().sum() < 2 or values.nunique(dropna=True) < 2:
        return pd.Series("all", index=series.index, dtype="object")
    try:
        binned = pd.qcut(values, q=min(q, values.nunique(dropna=True)), duplicates="drop")
        return binned.astype("string").fillna("missing")
    except Exception:
        return pd.Series("all", index=series.index, dtype="object")


def add_operating_regime_bins(frame):
    out = frame.copy()
    out["target_power_bin"] = pd.cut(
        out["y_true"].clip(lower=0, upper=1),
        bins=POWER_BINS,
        labels=POWER_BIN_LABELS,
        include_lowest=True,
    ).astype("string")
    out["predicted_power_bin"] = pd.cut(
        out["y_pred"].clip(lower=0, upper=1),
        bins=POWER_BINS,
        labels=POWER_BIN_LABELS,
        include_lowest=True,
    ).astype("string")
    if "nwp_fcst_horiz_hours" in out.columns:
        out["nwp_forecast_horizon_bin"] = qbin_with_fallback(out["nwp_fcst_horiz_hours"], q=4)
    if "wind_speed_ms" in out.columns and out["wind_speed_ms"].notna().any():
        out["wind_speed_bin"] = qbin_with_fallback(out["wind_speed_ms"], q=5)
    if "T_HAG_2_M" in out.columns:
        out["temperature_bin"] = qbin_with_fallback(out["T_HAG_2_M"], q=4)
    return out


def regime_metric_table(frame, regime_column, regime_type):
    rows = []
    if regime_column not in frame.columns:
        return rows
    for regime_bin, group in frame.groupby(regime_column, dropna=False, sort=True):
        if len(group) == 0:
            continue
        metrics = regression_metrics(group["y_true"], group["y_pred"])
        rows.append({
            "regime_type": regime_type,
            "regime_bin": str(regime_bin),
            "n_rows": int(len(group)),
            "MAE": metrics["MAE"],
            "RMSE": metrics["RMSE"],
            "mean_residual": float(group["residual"].mean()),
            "p95_abs_error": safe_quantile(group["abs_error"], 0.95),
            "warning_rate": float(group["warning_any"].mean()),
        })
    return rows


test_residual_df = add_operating_regime_bins(test_residual_df)
val_residual_df = add_operating_regime_bins(val_residual_df)

regime_columns = [
    ("target_power_bin", "target_power"),
    ("predicted_power_bin", "predicted_power"),
    ("nwp_forecast_horizon_bin", "nwp_forecast_horizon"),
    ("wind_speed_bin", "wind_speed"),
    ("temperature_bin", "temperature"),
]

operating_regime_rows = []
for column, regime_type in regime_columns:
    operating_regime_rows.extend(regime_metric_table(test_residual_df, column, regime_type))

operating_regime_summary_df = pd.DataFrame(operating_regime_rows)
if not operating_regime_summary_df.empty:
    operating_regime_summary_df = operating_regime_summary_df.sort_values(
        ["regime_type", "MAE"], ascending=[True, False]
    ).reset_index(drop=True)

display(operating_regime_summary_df)


## Temporal Residual Diagnostics

Τα temporal diagnostics συνοψίζουν residual behavior ανά ώρα, μήνα και quarter/season όταν το timestamp είναι διαθέσιμο. Είναι περιγραφικά και δεν εισάγουν νέα thresholds.


In [ ]:
# ============================================================
# NB20 | Temporal residual diagnostics
# ============================================================

def add_temporal_columns(frame):
    out = frame.copy()
    timestamp = pd.to_datetime(out[TIME_COL], errors="coerce")
    out["hour"] = timestamp.dt.hour
    out["day_of_week"] = timestamp.dt.dayofweek
    out["month"] = timestamp.dt.month
    out["quarter"] = timestamp.dt.quarter
    season_map = {
        12: "winter",
        1: "winter",
        2: "winter",
        3: "spring",
        4: "spring",
        5: "spring",
        6: "summer",
        7: "summer",
        8: "summer",
        9: "autumn",
        10: "autumn",
        11: "autumn",
    }
    out["season"] = out["month"].map(season_map)
    return out


def residual_metric_summary(group):
    metrics = regression_metrics(group["y_true"], group["y_pred"])
    return {
        "n_rows": int(len(group)),
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "mean_residual": float(group["residual"].mean()),
        "p95_abs_error": safe_quantile(group["abs_error"], 0.95),
        "warning_rate": float(group["warning_flag"].mean()),
    }


def temporal_metric_table(frame, split_name, temporal_column):
    rows = []
    if temporal_column not in frame.columns:
        return rows
    for temporal_value, group in frame.groupby(temporal_column, dropna=False, sort=True):
        if len(group) == 0:
            continue
        rows.append({
            "split": split_name,
            "temporal_dimension": temporal_column,
            "temporal_value": str(temporal_value),
            **residual_metric_summary(group),
        })
    return rows


val_residual_df = add_temporal_columns(val_residual_df)
test_residual_df = add_temporal_columns(test_residual_df)

temporal_rows = []
for split_name, frame in [("validation", val_residual_df), ("test", test_residual_df)]:
    for temporal_column in ["hour", "month", "quarter", "season"]:
        temporal_rows.extend(temporal_metric_table(frame, split_name, temporal_column))

temporal_summary_df = pd.DataFrame(temporal_rows)
display(temporal_summary_df)


## Park Metadata Relationship Diagnostics

Αν υπάρχουν `lat`/`long`, δημιουργείται map-like scatter plot με residual metric ανά park. Αν υπάρχουν turbine metadata, υπολογίζονται απλές συσχετίσεις με residual metrics χωρίς causal interpretation.


In [ ]:
# ============================================================
# NB20 | Spatial / metadata residual layer
# ============================================================

metadata_columns_for_relationships = [
    col for col in ["lat", "long", "hub_height_m", "rotor_diameter_m", "nominal_power_kW"]
    if col in park_level_summary_df.columns
]
residual_metric_columns = ["MAE", "warning_rate", "abs_mean_residual", "event_count", "longest_event_duration"]

correlation_rows = []
for metadata_col in metadata_columns_for_relationships:
    for metric_col in residual_metric_columns:
        pair = park_level_summary_df[[metadata_col, metric_col]].apply(pd.to_numeric, errors="coerce").dropna()
        if len(pair) >= 3 and pair[metadata_col].nunique() > 1 and pair[metric_col].nunique() > 1:
            correlation_rows.append({
                "metadata_column": metadata_col,
                "residual_metric": metric_col,
                "pearson_correlation": float(pair[metadata_col].corr(pair[metric_col])),
                "n_parks": int(len(pair)),
                "interpretation_boundary": "descriptive_correlation_no_causal_claim",
            })

metadata_correlation_df = pd.DataFrame(correlation_rows)
metadata_correlation_summary_df = metadata_correlation_df.copy()

display(metadata_correlation_summary_df)


## Optional Existing Model-Comparison Residual Analysis

Τα NB06/NB07 prediction CSVs, όταν έχουν συμβατό schema, χρησιμοποιούνται μόνο για test residual interpretation ανά model. Δεν χρησιμοποιούνται για validation thresholds ή model selection.


In [ ]:
# ============================================================
# NB20 | Optional model-comparison residual interpretation
# ============================================================

def first_existing_column(columns, candidates):
    for candidate in candidates:
        if candidate in columns:
            return candidate
    return None


def load_prediction_csv_for_comparison(source_name, path):
    if not path.exists():
        return pd.DataFrame(), {"source_name": source_name, "loaded": False, "reason": "missing"}
    try:
        read_kwargs = {}
        if SMOKE_MODE and not RUN_FULL_DIAGNOSTICS:
            read_kwargs["nrows"] = SMOKE_OPTIONAL_PREDICTION_MAX_ROWS
        df = pd.read_csv(path, **read_kwargs)
    except Exception as exc:
        return pd.DataFrame(), {"source_name": source_name, "loaded": False, "reason": f"read_error: {exc}"}

    park_col = first_existing_column(df.columns, ["park_id", "park", "plant_id"])
    time_col = first_existing_column(df.columns, ["timestamp", "time", "datetime"])
    truth_col = first_existing_column(df.columns, ["y_true", TARGET_COL, "target", "actual"])
    pred_col = first_existing_column(df.columns, ["y_pred", "prediction", "predicted", "model_prediction"])
    model_col = first_existing_column(df.columns, ["model", "model_name", "source"])

    missing = [name for name, value in {
        "park": park_col,
        "timestamp": time_col,
        "truth": truth_col,
        "prediction": pred_col,
    }.items() if value is None]
    if missing:
        return pd.DataFrame(), {"source_name": source_name, "loaded": False, "reason": "missing_columns:" + ",".join(missing)}

    out = df[[park_col, time_col, truth_col, pred_col] + ([model_col] if model_col else [])].copy()
    out = out.rename(columns={park_col: PARK_COL, time_col: TIME_COL, truth_col: "y_true", pred_col: "y_pred"})
    if model_col is None:
        out["model"] = source_name
    else:
        out = out.rename(columns={model_col: "model"})
    out[PARK_COL] = out[PARK_COL].map(normalize_park_id_value).astype("string")
    out[TIME_COL] = pd.to_datetime(out[TIME_COL], errors="coerce")
    out["y_true"] = pd.to_numeric(out["y_true"], errors="coerce")
    out["y_pred"] = pd.to_numeric(out["y_pred"], errors="coerce")
    out = out.dropna(subset=[PARK_COL, TIME_COL, "y_true", "y_pred", "model"]).copy()
    out["source_file"] = source_name
    smoke_limited = bool(SMOKE_MODE and not RUN_FULL_DIAGNOSTICS)
    return out, {
        "source_name": source_name,
        "loaded": True,
        "reason": "ok_smoke_limited" if smoke_limited else "ok",
        "rows": len(out),
        "models": out["model"].nunique(),
        "smoke_limited": smoke_limited,
    }


comparison_frames = []
comparison_load_audit_rows = []
for source_name, path in OPTIONAL_PREDICTION_PATHS.items():
    frame, audit = load_prediction_csv_for_comparison(source_name, path)
    comparison_load_audit_rows.append(audit)
    if not frame.empty:
        comparison_frames.append(frame)

model_prediction_load_audit_df = pd.DataFrame(comparison_load_audit_rows)

if comparison_frames:
    all_model_predictions_df = pd.concat(comparison_frames, ignore_index=True)
    all_model_predictions_df = all_model_predictions_df.drop_duplicates(
        subset=["source_file", "model", PARK_COL, TIME_COL]
    )
    all_model_predictions_df["residual"] = all_model_predictions_df["y_true"] - all_model_predictions_df["y_pred"]
    all_model_predictions_df["abs_error"] = all_model_predictions_df["residual"].abs()
    all_model_predictions_df["squared_error"] = all_model_predictions_df["residual"] ** 2

    comparison_rows = []
    for (source_file, model), group in all_model_predictions_df.groupby(["source_file", "model"], sort=True):
        metrics = regression_metrics(group["y_true"], group["y_pred"])
        comparison_rows.append({
            "source_file": source_file,
            "model": model,
            "n_rows": int(len(group)),
            "parks": int(group[PARK_COL].nunique()),
            "MAE": metrics["MAE"],
            "RMSE": metrics["RMSE"],
            "R2": metrics["R2"],
            "mean_residual": float(group["residual"].mean()),
            "p95_abs_error": safe_quantile(group["abs_error"], 0.95),
        })
    model_residual_comparison_df = pd.DataFrame(comparison_rows).sort_values(["MAE", "RMSE"], ascending=[True, True]).reset_index(drop=True)
else:
    all_model_predictions_df = pd.DataFrame()
    model_residual_comparison_df = pd.DataFrame()

display(model_prediction_load_audit_df)
display(model_residual_comparison_df)


## Figures

Οι figures είναι matplotlib-only και αποθηκεύονται μόνο όταν `EXPORT_RESULTS=True`. Με default flags εμφανίζονται local στο notebook run αλλά δεν γράφονται στον δίσκο.


In [ ]:
# ============================================================
# NB20 | Manuscript-friendly local figures
# ============================================================

figures_created = []
figures_saved = []


def save_or_register_figure(fig, filename):
    figures_created.append(filename)
    if EXPORT_RESULTS:
        FIGURE_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(FIGURE_DIR / filename, dpi=160, bbox_inches="tight")
        figures_saved.append(filename)


def top_parks_by(metric, n=3):
    if park_level_summary_df.empty or metric not in park_level_summary_df.columns:
        return []
    return park_level_summary_df.sort_values(metric, ascending=False)[PARK_COL].head(n).tolist()


fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(val_residual_df["residual"].dropna(), bins=40, alpha=0.55, label="validation", density=True)
ax.hist(test_residual_df["residual"].dropna(), bins=40, alpha=0.55, label="test", density=True)
ax.axvline(0, color="black", linewidth=1)
ax.set_title("Residual distribution: validation vs test")
ax.set_xlabel("Residual (observed - predicted normalized power)")
ax.set_ylabel("Density")
ax.legend()
fig.tight_layout()
save_or_register_figure(fig, "residual_distribution_validation_test.png")
plt.show()

plot_df = park_level_summary_df.sort_values("MAE", ascending=False).head(TOP_N_DISPLAY).sort_values("MAE")
fig, ax = plt.subplots(figsize=(9, max(4, 0.35 * len(plot_df))))
ax.barh(plot_df[PARK_COL], plot_df["MAE"], color="#4C78A8")
ax.set_title("Top parks by test residual MAE")
ax.set_xlabel("MAE")
ax.set_ylabel("park_id")
fig.tight_layout()
save_or_register_figure(fig, "park_level_mae_top20.png")
plt.show()

plot_df = park_level_summary_df.sort_values("warning_rate", ascending=False).head(TOP_N_DISPLAY).sort_values("warning_rate")
fig, ax = plt.subplots(figsize=(9, max(4, 0.35 * len(plot_df))))
ax.barh(plot_df[PARK_COL], plot_df["warning_rate"], color="#F58518")
ax.set_title("Top parks by residual warning rate")
ax.set_xlabel("Warning rate")
ax.set_ylabel("park_id")
fig.tight_layout()
save_or_register_figure(fig, "park_level_warning_rate_top20.png")
plt.show()

top3_parks = top_diagnostic_parks_df[PARK_COL].head(3).tolist() if not top_diagnostic_parks_df.empty else []
fig, ax = plt.subplots(figsize=(11, 5))
for park_id in top3_parks:
    group = test_residual_df[test_residual_df[PARK_COL] == park_id]
    ax.plot(group[TIME_COL], group["rolling_MAE_24"], label=f"park {park_id}", linewidth=1.4)
ax.set_title("Rolling residual MAE-24 for top diagnostic parks")
ax.set_xlabel("Timestamp")
ax.set_ylabel("Rolling MAE-24")
ax.legend(loc="best")
fig.autofmt_xdate()
fig.tight_layout()
save_or_register_figure(fig, "rolling_residual_top3_parks.png")
plt.show()

fig, ax = plt.subplots(figsize=(11, 4.8))
if top3_parks:
    y_positions = {park_id: idx for idx, park_id in enumerate(top3_parks)}
    for park_id in top3_parks:
        group = test_residual_df[(test_residual_df[PARK_COL] == park_id) & (test_residual_df["warning_any"])]
        ax.scatter(group[TIME_COL], [y_positions[park_id]] * len(group), s=12, label=f"park {park_id}")
    ax.set_yticks(list(y_positions.values()))
    ax.set_yticklabels(list(y_positions.keys()))
else:
    ax.text(0.5, 0.5, "No warning events", ha="center", va="center", transform=ax.transAxes)
ax.set_title("Warning-event timeline for top diagnostic parks")
ax.set_xlabel("Timestamp")
ax.set_ylabel("park_id")
fig.autofmt_xdate()
fig.tight_layout()
save_or_register_figure(fig, "warning_event_timeline_top3_parks.png")
plt.show()

if not operating_regime_summary_df.empty:
    plot_df = operating_regime_summary_df.sort_values("MAE", ascending=False).head(18).sort_values("MAE")
    labels = plot_df["regime_type"] + ": " + plot_df["regime_bin"]
else:
    plot_df = pd.DataFrame({"MAE": [0.0]})
    labels = pd.Series(["no regimes"])
fig, ax = plt.subplots(figsize=(10, max(4, 0.35 * len(plot_df))))
ax.barh(labels, plot_df["MAE"], color="#54A24B")
ax.set_title("Operating regimes ranked by residual MAE")
ax.set_xlabel("MAE")
ax.set_ylabel("Regime")
fig.tight_layout()
save_or_register_figure(fig, "operating_regime_abs_error.png")
plt.show()

scatter_df = test_residual_df.sample(n=min(8000, len(test_residual_df)), random_state=SEED) if len(test_residual_df) > 0 else test_residual_df
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(scatter_df["y_pred"], scatter_df["residual"], s=8, alpha=0.35, color="#B279A2")
ax.axhline(0, color="black", linewidth=1)
ax.set_title("Residuals versus predicted normalized power")
ax.set_xlabel("Predicted normalized power")
ax.set_ylabel("Residual")
fig.tight_layout()
save_or_register_figure(fig, "residual_vs_predicted_power.png")
plt.show()

if {"lat", "long"}.issubset(park_level_summary_df.columns) and park_level_summary_df[["lat", "long"]].notna().all(axis=1).any():
    spatial_df = park_level_summary_df.dropna(subset=["lat", "long"]).copy()
    fig, ax = plt.subplots(figsize=(7.5, 6))
    points = ax.scatter(spatial_df["long"], spatial_df["lat"], c=spatial_df["MAE"], s=55, cmap="viridis", edgecolor="black", linewidth=0.3)
    ax.set_title("Park residual MAE by location")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    fig.colorbar(points, ax=ax, label="MAE")
    fig.tight_layout()
    save_or_register_figure(fig, "spatial_residual_map_if_latlong.png")
    plt.show()
else:
    spatial_df = pd.DataFrame()


if EXPORT_RESULTS and "temporal_summary_df" in globals() and not temporal_summary_df.empty:
    hour_warning_df = temporal_summary_df.query("split == 'test' and temporal_dimension == 'hour'").copy()
    if not hour_warning_df.empty:
        hour_warning_df["temporal_value_numeric"] = pd.to_numeric(hour_warning_df["temporal_value"], errors="coerce")
        hour_warning_df = hour_warning_df.sort_values("temporal_value_numeric")
        fig, ax = plt.subplots(figsize=(9, 4.5))
        ax.plot(hour_warning_df["temporal_value_numeric"], hour_warning_df["warning_rate"], marker="o", color="#E45756")
        ax.set_title("Test warning rate by hour")
        ax.set_xlabel("Hour")
        ax.set_ylabel("Warning rate")
        ax.set_xticks(sorted(hour_warning_df["temporal_value_numeric"].dropna().unique()))
        fig.tight_layout()
        save_or_register_figure(fig, "temporal_warning_rate_by_hour.png")
        plt.show()

    month_abs_error_df = temporal_summary_df.query("split == 'test' and temporal_dimension == 'month'").copy()
    if not month_abs_error_df.empty:
        month_abs_error_df["temporal_value_numeric"] = pd.to_numeric(month_abs_error_df["temporal_value"], errors="coerce")
        month_abs_error_df = month_abs_error_df.sort_values("temporal_value_numeric")
        fig, ax = plt.subplots(figsize=(9, 4.5))
        ax.bar(month_abs_error_df["temporal_value_numeric"].astype(str), month_abs_error_df["MAE"], color="#72B7B2")
        ax.set_title("Test residual MAE by month")
        ax.set_xlabel("Month")
        ax.set_ylabel("MAE")
        fig.tight_layout()
        save_or_register_figure(fig, "temporal_abs_error_by_month.png")
        plt.show()

if EXPORT_RESULTS and "metadata_correlation_summary_df" in globals() and not park_level_summary_df.empty:
    metadata_plot_candidates = [col for col in ["nominal_power_kW", "hub_height_m", "rotor_diameter_m"] if col in park_level_summary_df.columns]
    if metadata_plot_candidates:
        metadata_col = metadata_plot_candidates[0]
        plot_df = park_level_summary_df[[metadata_col, "warning_rate"]].apply(pd.to_numeric, errors="coerce").dropna()
        if len(plot_df) >= 3 and plot_df[metadata_col].nunique() > 1:
            fig, ax = plt.subplots(figsize=(7, 4.8))
            ax.scatter(plot_df[metadata_col], plot_df["warning_rate"], s=55, color="#F58518", edgecolor="black", linewidth=0.3)
            ax.set_title(f"Warning rate versus {metadata_col}")
            ax.set_xlabel(metadata_col)
            ax.set_ylabel("Warning rate")
            fig.tight_layout()
            save_or_register_figure(fig, "metadata_vs_warning_rate.png")
            plt.show()

figure_audit_df = pd.DataFrame({
    "figure": figures_created,
    "saved": [name in figures_saved for name in figures_created],
})
display(figure_audit_df)


## Manuscript Table Builder

Οι παρακάτω πίνακες είναι συμπυκνωμένοι για manuscript drafting. Τα full local artifacts γράφονται μόνο με `EXPORT_RESULTS=True`.


In [ ]:
# ============================================================
# NB20 | Manuscript-ready tables
# ============================================================

def overall_metric_row(split_name, frame):
    metrics = regression_metrics(frame["y_true"], frame["y_pred"])
    return {
        "split": split_name,
        "n_rows": int(len(frame)),
        "parks": int(frame[PARK_COL].nunique()),
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "R2": metrics["R2"],
        "mean_residual": float(frame["residual"].mean()),
        "median_abs_error": float(frame["abs_error"].median()),
        "p95_abs_error": safe_quantile(frame["abs_error"], 0.95),
        "warning_rate": float(frame["warning_flag"].mean()) if "warning_flag" in frame.columns else np.nan,
    }


table_A_overall_validation_test_residual_metrics_df = pd.DataFrame([
    overall_metric_row("validation", val_residual_df),
    overall_metric_row("test", test_residual_df),
])
table_residual_overall_metrics_df = table_A_overall_validation_test_residual_metrics_df.copy()

table_B_top10_diagnostic_parks_by_MAE_df = park_level_summary_df.sort_values(
    ["MAE", "warning_rate", "abs_mean_residual"], ascending=[False, False, False]
).head(10)[[
    "diagnostic_rank",
    "park_id",
    "n_test_rows",
    "MAE",
    "RMSE",
    "warning_rate",
    "event_count",
    "longest_event_duration",
    "mean_residual",
    "p95_abs_error",
]].copy()
table_top_diagnostic_parks_df = table_B_top10_diagnostic_parks_by_MAE_df.copy()

table_C_top10_diagnostic_parks_by_warning_rate_df = park_level_summary_df.sort_values(
    ["warning_rate", "MAE", "abs_mean_residual"], ascending=[False, False, False]
).head(10)[[
    "diagnostic_rank",
    "park_id",
    "n_test_rows",
    "MAE",
    "warning_rate",
    "event_count",
    "longest_event_duration",
    "mean_residual",
]].copy()

table_D_top10_warning_events_by_severity_df = top_warning_events_df.head(10).copy()

table_E_operating_regime_top_high_error_regimes_df = operating_regime_summary_df.sort_values(
    ["MAE", "warning_rate"], ascending=[False, False]
).head(10).copy() if not operating_regime_summary_df.empty else pd.DataFrame()

table_warning_events_df = warning_event_summary_df.head(25).copy()
table_operating_regime_summary_df = operating_regime_summary_df.head(30).copy()
table_model_residual_comparison_df = model_residual_comparison_df.copy()

manuscript_table_registry_df = pd.DataFrame([
    {"table_id": "Table A", "object_name": "table_A_overall_validation_test_residual_metrics_df", "description": "overall validation/test residual metrics", "rows": len(table_A_overall_validation_test_residual_metrics_df)},
    {"table_id": "Table B", "object_name": "table_B_top10_diagnostic_parks_by_MAE_df", "description": "top 10 diagnostic parks by MAE", "rows": len(table_B_top10_diagnostic_parks_by_MAE_df)},
    {"table_id": "Table C", "object_name": "table_C_top10_diagnostic_parks_by_warning_rate_df", "description": "top 10 diagnostic parks by warning rate", "rows": len(table_C_top10_diagnostic_parks_by_warning_rate_df)},
    {"table_id": "Table D", "object_name": "table_D_top10_warning_events_by_severity_df", "description": "top 10 warning events by severity", "rows": len(table_D_top10_warning_events_by_severity_df)},
    {"table_id": "Table E", "object_name": "table_E_operating_regime_top_high_error_regimes_df", "description": "operating-regime summary top high-error regimes", "rows": len(table_E_operating_regime_top_high_error_regimes_df)},
])

display(manuscript_table_registry_df)
display(table_A_overall_validation_test_residual_metrics_df)
display(table_B_top10_diagnostic_parks_by_MAE_df)
display(table_C_top10_diagnostic_parks_by_warning_rate_df)
display(table_D_top10_warning_events_by_severity_df)
display(table_E_operating_regime_top_high_error_regimes_df)
if not table_model_residual_comparison_df.empty:
    display(table_model_residual_comparison_df)


## Manuscript Figure Caption Drafts

Οι caption drafts είναι templates για manuscript writing. Δεν υπονοούν confirmed faults και επαναλαμβάνουν το leakage boundary.

- **Residual distributions for validation and test splits.** The figure shows residual distributions derived from the selected forecasting source. Validation residuals are used only to derive thresholds, while test residuals are used for one-time diagnostic interpretation. Heavy tails or shifted distributions indicate systematic deviations from expected forecasting behavior, but do not by themselves confirm physical faults.
- **Top parks by residual MAE.** The figure ranks test parks by mean absolute residual after validation-calibrated thresholds are fixed. It supports condition-monitoring-oriented prioritization, not fault diagnosis.
- **Top parks by warning rate.** The figure summarizes how often each park crosses validation-derived residual warning rules on the test split. Elevated warning rate indicates persistent deviation from expected forecast behavior and should be interpreted as early-warning evidence only.
- **Rolling residual MAE for top diagnostic parks.** The figure shows rolling MAE-24 trajectories for high-ranked parks. Rolling behavior emphasizes persistence and reduces emphasis on isolated spikes.
- **Warning-event timeline for top diagnostic parks.** The figure marks contiguous warning periods derived from fixed validation thresholds. These are candidate diagnostic events, not labeled failures.
- **Operating regimes ranked by residual MAE.** The figure shows regimes where residual stress is concentrated. It supports discussion of operating-regime mismatch without causal claims.
- **Residuals versus predicted power.** The figure checks whether residual stress concentrates at particular predicted output levels. Patterns can indicate model mismatch under certain expected power states.
- **Spatial residual map.** When latitude/longitude are available, the figure maps park-level residual metrics. Spatial clustering is descriptive and does not prove physical causes.
- **Temporal warning rate by hour.** When exported, the figure summarizes hourly warning-rate variation using fixed validation-derived warning rules.
- **Temporal residual MAE by month.** When exported, the figure summarizes monthly residual magnitude and supports seasonal or temporal discussion without changing thresholds.
- **Metadata versus warning rate.** When exported, the figure compares available park metadata with warning rate as descriptive evidence only.


## Local-Only Exports

Τα exports γράφονται κάτω από `data/processed/diagnostics/residual_phm_diagnostics/` μόνο όταν `EXPORT_RESULTS=True`. Δεν γράφονται model binaries, checkpoints ή αλλαγές σε canonical benchmark files.


In [ ]:
# ============================================================
# NB20 | Local-only exports με explicit policy
# ============================================================

exported_csv_paths = []

run_manifest_df = pd.DataFrame([
    {"key": "notebook", "value": "20_residual_phm_diagnostics.ipynb"},
    {"key": "seed", "value": SEED},
    {"key": "SMOKE_MODE", "value": SMOKE_MODE},
    {"key": "RUN_FULL_DIAGNOSTICS", "value": RUN_FULL_DIAGNOSTICS},
    {"key": "EXPORT_RESULTS", "value": EXPORT_RESULTS},
    {"key": "FULL_EXPORT_RESIDUAL_RECORDS", "value": FULL_EXPORT_RESIDUAL_RECORDS},
    {"key": "selected_prediction_source", "value": selected_prediction_source["label"]},
    {"key": "validation_threshold_source", "value": selected_prediction_source["validation_threshold_source"]},
    {"key": "test_evaluation_count", "value": test_evaluation_count},
    {"key": "test_parks", "value": test_residual_df[PARK_COL].nunique()},
    {"key": "warning_events", "value": len(warning_event_summary_df)},
    {"key": "output_dir", "value": str(OUTPUT_DIR.relative_to(PROJECT_ROOT))},
])


def export_csv(df, filename):
    if EXPORT_RESULTS:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        path = OUTPUT_DIR / filename
        df.to_csv(path, index=False)
        exported_csv_paths.append(path)


export_csv(run_manifest_df, "residual_phm_run_manifest.csv")
export_csv(path_audit_df, "residual_phm_path_audit.csv")
export_csv(prediction_source_audit_df, "residual_phm_prediction_source_audit.csv")
export_csv(threshold_policy_df, "residual_phm_threshold_policy.csv")
export_csv(table_residual_overall_metrics_df, "residual_phm_overall_metrics.csv")
export_csv(park_level_summary_df, "residual_phm_park_level_summary.csv")
export_csv(table_top_diagnostic_parks_df, "residual_phm_top_diagnostic_parks.csv")
export_csv(warning_event_summary_df, "residual_phm_warning_event_summary.csv")
export_csv(operating_regime_summary_df, "residual_phm_operating_regime_summary.csv")
export_csv(directional_bias_summary_df, "residual_phm_directional_bias_summary.csv")
export_csv(temporal_summary_df, "residual_phm_temporal_summary.csv")
export_csv(residual_persistence_summary_df, "residual_phm_residual_persistence_summary.csv")
export_csv(top_warning_events_df, "residual_phm_top_warning_events.csv")
export_csv(metadata_correlation_summary_df, "residual_phm_metadata_correlation_summary.csv")

if not model_residual_comparison_df.empty:
    export_csv(model_residual_comparison_df, "residual_phm_model_comparison.csv")

sample_size = min(10000, len(test_residual_df))
test_residual_sample_df = test_residual_df.sample(n=sample_size, random_state=SEED) if sample_size > 0 else test_residual_df.copy()
export_csv(test_residual_sample_df, "residual_phm_test_residual_records_sample.csv")

if EXPORT_RESULTS and FULL_EXPORT_RESIDUAL_RECORDS:
    export_csv(test_residual_df, "residual_phm_test_residual_records_full.csv")

export_audit_df = pd.DataFrame({
    "exported_csv": [str(path.relative_to(PROJECT_ROOT)) for path in exported_csv_paths],
})

display(run_manifest_df)
display(export_audit_df)


## Final Self-Checks

Τα checks κλειδώνουν το leakage policy, την export policy και το manuscript boundary πριν θεωρηθεί ολοκληρωμένο το NB20.


In [ ]:
# ============================================================
# NB20 | Τελικοί έλεγχοι συνέπειας και boundary
# ============================================================

def protected_path_unchanged(name):
    path = PROTECTED_PATHS[name]
    before = protected_mtimes_before[name]
    after = path.stat().st_mtime_ns if path.exists() else None
    return before == after


def no_model_binary_outputs_created():
    if not OUTPUT_DIR.exists():
        return True
    return not any(path.suffix.lower() in FORBIDDEN_OUTPUT_SUFFIXES for path in OUTPUT_DIR.rglob("*"))


def required_columns_present(frame, columns):
    return all(column in frame.columns for column in columns)


def warning_events_respect_timestamp_gaps():
    if warning_event_summary_df.empty:
        return True
    required_gap_columns = ["duration_hours_if_inferable", "timestamp_gap_breaks_used"]
    if not required_columns_present(warning_event_summary_df, required_gap_columns):
        return False
    if "warning_event_interval_audit_df" not in globals() or warning_event_interval_audit_df.empty:
        return False
    return warning_event_summary_df["timestamp_gap_breaks_used"].notna().all()


def fallback_train_validation_test_audit_ok():
    fallback_used = bool(selected_prediction_source.get("requires_fallback_model", False))
    if not fallback_used:
        return "fallback_train_validation_test_audit_df" in globals() and not fallback_train_validation_test_audit_df.empty
    required_columns = [
        "train_MAE",
        "train_RMSE",
        "train_R2",
        "validation_MAE",
        "validation_RMSE",
        "validation_R2",
        "test_MAE",
        "test_RMSE",
        "test_R2",
        "train_validation_gap_MAE",
        "validation_test_gap_MAE",
    ]
    return (
        "fallback_train_validation_test_audit_df" in globals()
        and not fallback_train_validation_test_audit_df.empty
        and required_columns_present(fallback_train_validation_test_audit_df, required_columns)
    )


def prediction_provenance_recorded():
    return (
        "prediction_provenance_status" in selected_prediction_source
        and "prediction_provenance_note" in selected_prediction_source
        and "prediction_provenance_status" in prediction_source_audit_df.columns
        and "prediction_provenance_note" in prediction_source_audit_df.columns
    )


def warning_event_columns_ok():
    if warning_event_summary_df.empty:
        return True
    required_event_columns = [
        "event_id",
        "park_id",
        "event_start",
        "event_end",
        "duration_rows",
        "max_abs_error",
        "mean_abs_error",
        "max_rolling_MAE_24",
        "mean_residual",
        "dominant_warning_reason",
        "severity_score",
        "duration_hours_if_inferable",
        "timestamp_gap_breaks_used",
    ]
    core_non_null_columns = [
        "event_id",
        "park_id",
        "event_start",
        "event_end",
        "duration_rows",
        "max_abs_error",
        "mean_abs_error",
        "mean_residual",
        "dominant_warning_reason",
        "severity_score",
    ]
    return (
        required_columns_present(warning_event_summary_df, required_event_columns)
        and warning_event_summary_df[core_non_null_columns].notna().all().all()
    )


check_rows = [
    {"check": "train_path_exists", "status": TRAIN_PATH.exists(), "critical": True},
    {"check": "val_path_exists", "status": VAL_PATH.exists(), "critical": True},
    {"check": "test_path_exists", "status": TEST_PATH.exists(), "critical": True},
    {"check": "target_present", "status": TARGET_COL in train_df.columns and TARGET_COL in val_df.columns and TARGET_COL in test_df.columns, "critical": True},
    {"check": "park_id_present", "status": PARK_COL in train_df.columns and PARK_COL in val_df.columns and PARK_COL in test_df.columns, "critical": True},
    {"check": "timestamp_present", "status": TIME_COL in train_df.columns and TIME_COL in val_df.columns and TIME_COL in test_df.columns, "critical": True},
    {"check": "prediction_source_resolved", "status": prediction_source_resolved, "critical": True},
    {"check": "validation_thresholds_used", "status": validation_thresholds_used, "critical": True},
    {"check": "no_test_threshold_leakage", "status": no_test_threshold_leakage, "critical": True},
    {"check": "test_evaluations_expected_one", "status": test_evaluation_count == 1, "critical": True},
    {"check": "selected_prediction_not_constant", "status": bool(selected_prediction_not_constant), "critical": True},
    {"check": "selected_prediction_not_equal_target", "status": bool(selected_prediction_not_equal_target), "critical": True},
    {"check": "prediction_provenance_recorded", "status": prediction_provenance_recorded(), "critical": True},
    {"check": "warning_events_respect_timestamp_gaps", "status": warning_events_respect_timestamp_gaps(), "critical": True},
    {"check": "fallback_train_validation_test_audit_created_if_fallback_used", "status": fallback_train_validation_test_audit_ok(), "critical": True},
    {"check": "residual_z_score_present", "status": required_columns_present(val_residual_df, ["residual_z_score"]) and required_columns_present(test_residual_df, ["residual_z_score"]), "critical": True},
    {"check": "robust_residual_z_score_present", "status": required_columns_present(val_residual_df, ["robust_residual_z_score"]) and required_columns_present(test_residual_df, ["robust_residual_z_score"]), "critical": True},
    {"check": "warning_flag_present", "status": required_columns_present(val_residual_df, ["warning_flag"]) and required_columns_present(test_residual_df, ["warning_flag"]), "critical": True},
    {"check": "validation_thresholds_used_for_test_warnings", "status": "rolling_MAE_24_q95" in test_residual_df.columns and "abs_error_q95" in test_residual_df.columns, "critical": True},
    {"check": "no_test_residuals_used_in_threshold_derivation", "status": no_test_threshold_leakage and set(threshold_policy_df["source_split"].dropna().unique()) == {"validation"}, "critical": True},
    {"check": "warning_event_summary_non_null_columns_present", "status": warning_event_columns_ok(), "critical": True},
    {"check": "temporal_diagnostics_created", "status": "temporal_summary_df" in globals() and not temporal_summary_df.empty, "critical": True},
    {"check": "directional_bias_diagnostics_created", "status": "directional_bias_summary_df" in globals() and not directional_bias_summary_df.empty, "critical": True},
    {"check": "persistence_diagnostics_created", "status": "residual_persistence_summary_df" in globals() and not residual_persistence_summary_df.empty, "critical": True},
    {"check": "manuscript_tables_created", "status": "manuscript_table_registry_df" in globals() and len(manuscript_table_registry_df) == 5, "critical": True},
    {"check": "no_forbidden_artifact_suffixes_written", "status": no_model_binary_outputs_created(), "critical": True},
    {"check": "outputs_not_required_for_scaffold_commit", "status": not EXPORT_RESULTS or len(exported_csv_paths) > 0, "critical": True},
    {"check": "baseline_metrics_not_modified", "status": protected_path_unchanged("baseline_metrics"), "critical": True},
    {"check": "requirements_not_modified", "status": protected_path_unchanged("requirements"), "critical": True},
    {"check": "no_model_binary_outputs", "status": no_model_binary_outputs_created(), "critical": True},
    {"check": "export_policy_ok", "status": bool((not EXPORT_RESULTS) or OUTPUT_DIR.exists()), "critical": True},
    {"check": "manuscript_boundary_ok", "status": True, "critical": True},
]

check_df = pd.DataFrame(check_rows)
display(check_df)

failed_critical_checks = check_df.query("critical == True and status == False")
if not failed_critical_checks.empty:
    raise ValueError("Critical NB20 self-checks failed: " + ", ".join(failed_critical_checks["check"].tolist()))


## Final Greek Summary

Η τελική cell συνοψίζει το selected prediction source, την validation threshold source, τα test parks, τα warning events και τα top diagnostic parks.


In [ ]:
# ============================================================
# NB20 | Τελική σύνοψη για manuscript log
# ============================================================

top3_summary_df = table_top_diagnostic_parks_df.head(3)[["park_id", "MAE", "warning_rate"]].copy()

print("Τελική σύνοψη NB20")
print(f"Selected prediction source: {selected_prediction_source['label']}")
print(f"Validation threshold source: {selected_prediction_source['validation_threshold_source']}")
print(f"Number of test parks: {test_residual_df[PARK_COL].nunique()}")
print(f"Number of warning events: {len(warning_event_summary_df)}")
print("Top 3 diagnostic parks by MAE/warning rate:")
if top3_summary_df.empty:
    print("  Δεν βρέθηκαν διαθέσιμα diagnostic parks.")
else:
    for _, row in top3_summary_df.iterrows():
        print(f"  park_id={row['park_id']} | MAE={row['MAE']:.6f} | warning_rate={row['warning_rate']:.4f}")
print(f"Number of exported CSVs: {len(exported_csv_paths)}")
print(f"Number of exported figures: {len(figures_saved)}")
print("Το NB20 ολοκληρώθηκε ως forecasting-based residual diagnostic layer, όχι ως confirmed fault diagnosis.")
